# Waypoint world model → TouchDesigner (experimental)
Open-source, image-seeded interactive video, targeting Colab A100 40 GB.
No Scope service or package is required. Model: Apache-2.0. Inference library: GPL-3.0.

Run the cells in order in a **fresh GPU runtime**. This notebook installs into a separate
environment; it does not install StreamDiffusion or replace the notebook kernel packages.
Setup creates the environment without ensurepip, then bootstraps pip into that interpreter.
Rerunning setup repairs a partial pip-less environment while retaining installed packages.
Do not run both GPU servers on the same GPU simultaneously.

1. Optional Drive storage reuses pinned model and VAE weights on later runtimes.
2. Launch with INT8 for Ampere/A100; BF16 is available for comparison.
3. Open the updated project/component in TouchDesigner, paste the URL and enable Connect.
   The handshake selects **World Model** and native **1280 × 720** output.
4. Choose an Input Source containing a scene image. Pulse **Seed World from Input**.
   The source is center-cropped to 16:9. First generation compiles kernels and can take minutes.
5. Set **World Button Codes** (comma-separated Owl-Control codes) and **Camera X/Y per Step**;
   values are held until changed. Set them to zero/empty to stop motion. Pause and Reset reuse weights.

This checkpoint has no direct text conditioning. For prompt-driven starting images, enable
**ENABLE_PROMPT_SEEDS** in Step 2 and choose **PROMPT_SEED_MODEL = sd-turbo / sdxl-turbo**.
In the updated TD component select **World Seed Source = Prompt Model (A/B)**.
The selected model generates a 768 × 432 seed inside Colab; Waypoint
animates it at native 720p. **Auto Reseed** defaults On, with an adjustable **5-second**
interval. Off keeps the current world until **Generate Seed from Prompt** / **Submit Prompts**.
Each reseed resets world history and can cause a scene cut. The timer starts after the first
usable batch and waits for output credit/batch boundaries; slow delivery can extend it.
Both models share one GPU worker. Combined GPU performance/memory use still need validation;
earlier Waypoint-only measurements do not establish this new mode's performance.
SDXL-Turbo blends both token and pooled text embeddings; only one seed model is loaded.
Its original VAE and both text encoders are retained for the seed pipeline. It uses more
GPU memory than SD-Turbo. Combined A100 memory, quality and speed need live validation.
Negative prompts, denoise strength, and audio reactivity do not steer prompt seeds.
Model selection applies at server restart. Existing cached weights are preserved.
Both seed models have separate Stability AI commercial-use terms; the bridge's MIT
license does not replace the model licenses: https://stability.ai/license.
Waypoint's native output is 720p, not 1024 × 768.
It generates four frames per controller step. GPU Inference shows the whole batch time.
Start with Fast video quality and Video Buffer 3. HTTP/2 is the default tunnel transport.
The default **fast** compiler profile keeps full-graph compilation but disables exhaustive
autotuning, coordinate-descent tuning, and CUDA-graph capture. **max-autotune** restores the
upstream policy for throughput comparisons after a successful first-frame test; it can take
much longer to start. Neither profile changes native resolution or quantization.
For INT8, fast also resets GemLite's independent tuner to its default after model construction.
Cold quantization warmup otherwise leaves exhaustive GemLite tuning enabled, even when
TorchInductor tuning is disabled. Initial quantization warmup and first-use compilation remain.
The updated TouchDesigner Status and server `/health` report seed preparation and first-batch
generation separately, with elapsed time. Model-loaded readiness does not mean a frame is ready.

Keep the Colab launch cell and TouchDesigner running. Interrupt the launch cell to stop its
server and tunnel. Drive persists weights, not the running GPU or compiled TensorRT engines.
Waypoint uses TorchInductor, not the SD-Turbo TensorRT cache. First-run compilation is still
required in a fresh runtime; cross-VM compiled-kernel reuse is not enabled here.

Basic native-720p INT8 generation was tested on an A100 40 GB with the fast profile and an
already-used VM/cache. Fresh-VM startup, long-running world quality, other profiles and sustained
TouchDesigner FPS still need validation. See the README measurements; this notebook does not
promise an A100 frame rate.

[Model](https://huggingface.co/Overworld/Waypoint-1.5-1B) ·
[Runtime and licenses](https://github.com/Overworldai/world_engine) ·
[Bridge source](https://github.com/wenjunii/colab-sd)


In [ ]:
# Step 1: Check the GPU
import subprocess
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_capability()[0] < 8:
    raise RuntimeError('Select an A100 GPU runtime (Ampere or newer is required).')
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
# @title Step 2: Install isolated Waypoint runtime and choose storage
# @markdown Repairs partial environment setup without ensurepip; keeps existing model caches.
USE_GOOGLE_DRIVE = False # @param {type:"boolean"}
DRIVE_MODEL_FOLDER = "StreamDiffusion/waypoint-models" # @param {type:"string"}
# @markdown Optional prompt-to-world pipeline. Both models share one GPU worker; new seeds reset the world.
ENABLE_PROMPT_SEEDS = False # @param {type:"boolean"}
PROMPT_SEED_MODEL = "sd-turbo" # @param ["sd-turbo", "sdxl-turbo"]

import os
import sys
from pathlib import Path
runtime_root = Path('/content/colab-sd-waypoint')
runtime_root.mkdir(exist_ok=True)
sources = {'colab_server.py': 'import os\nimport sys\nimport traceback\nimport gc\nimport hmac\n\nbackend_name = os.environ.get(\'BRIDGE_BACKEND\', \'sd-turbo\')\nis_world_backend = backend_name == \'waypoint\'\nis_krea_backend = backend_name == \'krea\'\nis_ltx_backend = backend_name == \'ltx\'\ndiffusion_model = os.environ.get(\'BRIDGE_DIFFUSION_MODEL\', \'sd-turbo\')\nif diffusion_model not in (\'sd-turbo\', \'sdxl-turbo\'):\n    raise ValueError(\'Diffusion model must be sd-turbo or sdxl-turbo\')\n\n\ndef model_label():\n    if is_ltx_backend:\n        return \'LTX-2.5 distilled 22B\'\n    if is_world_backend:\n        return \'Waypoint 1.5-1B\'\n    if is_krea_backend:\n        return \'Krea Realtime 14B\'\n    return \'SDXL-Turbo\' if diffusion_model == \'sdxl-turbo\' else \'SD-Turbo\'\n\nif not (is_world_backend or is_krea_backend or is_ltx_backend):\n    # 1. Compatibility shims for huggingface_hub\n    import huggingface_hub\n    import huggingface_hub.constants\n\n    if not hasattr(huggingface_hub.constants, \'hf_cache_home\'):\n        huggingface_hub.constants.hf_cache_home = getattr(\n            huggingface_hub.constants, \'HF_HOME\', os.path.expanduser(\'~/.cache/huggingface\')\n        )\n\n    if not hasattr(huggingface_hub, \'HfFolder\'):\n        class HfFolder:\n            @classmethod\n            def get_token(cls):\n                try:\n                    from huggingface_hub import get_token\n                    return get_token()\n                except Exception:\n                    return None\n            @classmethod\n            def save_token(cls, token):\n                pass\n        huggingface_hub.HfFolder = HfFolder\n\n    if not hasattr(huggingface_hub, \'cached_download\'):\n        from huggingface_hub import hf_hub_download\n        huggingface_hub.cached_download = hf_hub_download\n\n    # 2. Compatibility shims for transformers\n    import transformers\n    import transformers.utils\n\n    if not hasattr(transformers.utils, \'FLAX_WEIGHTS_NAME\'):\n        transformers.utils.FLAX_WEIGHTS_NAME = \'flax_model.msgpack\'\n    if not hasattr(transformers.utils, \'FLAX_WEIGHTS_INDEX_NAME\'):\n        transformers.utils.FLAX_WEIGHTS_INDEX_NAME = \'flax_model.msgpack.index.json\'\n\n    if not hasattr(transformers, \'EncoderDecoderCache\'):\n        class EncoderDecoderCache:\n            pass\n        transformers.EncoderDecoderCache = EncoderDecoderCache\n        try:\n            import transformers.cache_utils\n            transformers.cache_utils.EncoderDecoderCache = EncoderDecoderCache\n        except Exception:\n            pass\n\nimport time\nimport asyncio\nimport json\nimport logging\nimport threading\nfrom concurrent.futures import ThreadPoolExecutor\nimport cv2\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom fastapi import FastAPI, WebSocket, WebSocketDisconnect\nfrom fastapi.responses import HTMLResponse\nfrom fastapi.responses import JSONResponse\nfrom tensorrt_backend import compile_unet as compile_tensorrt_unet\nfrom bridge_protocol import (OUTPUT_RESOLUTIONS, PROTOCOL_VERSION, OUTPUT_CHUNK_BYTES,\n                             MAX_INPUT_FRAME_BYTES, VIDEO_QUALITY_LIMITS, FrameAssembler,\n                             resolution, validate_controls, video_byte_budget)\n\n# Disable autograd globally for fast inference\ntorch.set_grad_enabled(False)\nif torch.cuda.is_available():\n    torch.backends.cuda.matmul.allow_tf32 = True\n    torch.backends.cudnn.allow_tf32 = True\n    torch.backends.cudnn.benchmark = True\n\nsys.path.append(\'.\')\ntry:\n    from utils.wrapper import StreamDiffusionWrapper\n    from streamdiffusion.image_utils import postprocess_image\nexcept ImportError:\n    StreamDiffusionWrapper = None\n    postprocess_image = None\n\n# Patch image_utils.pt_to_numpy just in case\ntry:\n    import streamdiffusion.image_utils\n    orig_pt_to_numpy = streamdiffusion.image_utils.pt_to_numpy\n    def safe_pt_to_numpy(images):\n        return images.detach().cpu().permute(0, 2, 3, 1).float().numpy()\n    streamdiffusion.image_utils.pt_to_numpy = safe_pt_to_numpy\nexcept Exception:\n    pass\n\nlogging.basicConfig(level=logging.INFO)\nlogger = logging.getLogger("StreamDiffusionServer")\napp = FastAPI()\naccess_token = os.environ.get(\'BRIDGE_ACCESS_TOKEN\', \'\')\n\nstream_txt2img = None\nstream_img2img = None\npipeline_ready = False\n# The two modes share model weights and mutable state. Keep all GPU work on\n# one worker, including control updates, without blocking WebSocket handling.\nmodel_executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix="diffusion")\ntry:\n    max_stream_fps = float(os.environ.get("MAX_STREAM_FPS", "60"))\n    if not np.isfinite(max_stream_fps) or not 1 <= max_stream_fps <= 120:\n        raise ValueError()\nexcept ValueError:\n    logger.warning("Invalid MAX_STREAM_FPS; using 60")\n    max_stream_fps = 60.0\nprompt_embedding_cache = {}\napplied_prompt_state = None\ntelemetry_lock = threading.Lock()\ntelemetry_cache = (0.0, "--", "--")\nactive_client = None\nworld_backend = None\nkrea_backend = None\nltx_backend = None\n\n\ndef stateful_backend():\n    return ltx_backend if is_ltx_backend else krea_backend if is_krea_backend else world_backend\npipeline_revision = 0\nlast_pipeline_error = None\njpeg_quality_hints = {}\n\n# Engine acceleration state\nenable_tensorrt = os.environ.get("ENABLE_TENSORRT", "0") in ("1", "true", "True") or "--tensorrt" in sys.argv\nactive_engine = "PyTorch (FP16)"\nimg2img_engine = "PyTorch (FP16)"\n\n\ndef engine_for_mode(mode=None):\n    if is_ltx_backend:\n        from ltx_backend import LTXBackend\n        return ltx_backend.engine if ltx_backend is not None else LTXBackend.engine\n    if is_krea_backend:\n        if krea_backend is not None:\n            return krea_backend.engine\n        from krea_quant import int8_engine_label\n        return int8_engine_label(os.environ.get(\'KREA_INT8_PROFILE\', \'fast\'))\n    if is_world_backend:\n        return \'Waypoint (INT8)\' if os.environ.get(\'WAYPOINT_QUANT\', \'intw8a8\') == \'intw8a8\' else \'Waypoint (BF16)\'\n    if enable_tensorrt and not pipeline_ready:\n        return \'TensorRT (Unavailable)\' if last_pipeline_error else \'TensorRT (Preparing)\'\n    return img2img_engine if (mode or current_mode) == "img2img" else active_engine\n\n# Prompt state\ncurrent_prompt_a = "cinematic cyberpunk cityscape, neon lights, 8k, detailed"\ncurrent_prompt_b = "bioluminescent alien jungle, glowing flora, ethereal, 8k"\ncurrent_prompt_blend = 0.0\ncurrent_neg_prompt = "blurry, low quality, distorted, watermark"\n\ncurrent_mode = "txt2img"\ncurrent_seed = 42\ncurrent_strength = 0.6\noutput_width, output_height = ((512, 512) if is_ltx_backend else (832, 480) if is_krea_backend else (1280, 720) if is_world_backend else\n    resolution({\'width\': int(os.environ.get(\'BRIDGE_INITIAL_WIDTH\', \'512\')),\n                \'height\': int(os.environ.get(\'BRIDGE_INITIAL_HEIGHT\', \'512\'))}))\noutput_resolutions = OUTPUT_RESOLUTIONS\nif is_world_backend:\n    current_mode = \'world\'\nelif is_krea_backend:\n    current_mode = \'krea\'\nelif is_ltx_backend:\n    from ltx_backend import startup_size\n    current_mode = \'ltx\'\n    output_width, output_height = startup_size()\nallow_tensorrt_resize = os.environ.get(\'BRIDGE_TRT_ALLOW_RESIZE\', \'0\') == \'1\'\n\n\ndef resolution_locked():\n    return is_world_backend or is_krea_backend or is_ltx_backend or (enable_tensorrt and not allow_tensorrt_resize)\n\n\ndef requested_resolution(data):\n    if is_ltx_backend:\n        return output_width, output_height\n    if is_krea_backend:\n        return (832, 480)\n    if is_world_backend:\n        return (1280, 720)\n    return resolution(data, (output_width, output_height))\n\n\ndef validated_controls(data, current):\n    if is_ltx_backend:\n        from ltx_backend import validate_ltx_controls\n        return validate_ltx_controls(data, current)\n    if is_krea_backend:\n        from krea_backend import validate_krea_controls\n        return validate_krea_controls(data)\n    if is_world_backend:\n        from waypoint_backend import validate_world_controls\n        return validate_world_controls(data, current)\n    return validate_controls(data, current)\n\n\ndef world_progress():\n    if is_ltx_backend and ltx_backend is not None:\n        return ltx_backend.progress()\n    if is_krea_backend and krea_backend is not None:\n        return krea_backend.progress()\n    return world_backend.progress() if is_world_backend and world_backend is not None else {}\n\ndef set_seed(s):\n    torch.manual_seed(s)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(s)\n\ndef apply_prompts_and_blend(prompt_a=None, prompt_b=None, blend=None, neg_prompt=None):\n    global current_prompt_a, current_prompt_b, current_prompt_blend, current_neg_prompt\n    global applied_prompt_state\n    if prompt_a is not None and str(prompt_a).strip():\n        current_prompt_a = str(prompt_a).strip()\n    if prompt_b is not None:\n        current_prompt_b = str(prompt_b).strip()\n    if blend is not None:\n        current_prompt_blend = max(0.0, min(1.0, float(blend)))\n    if neg_prompt is not None:\n        current_neg_prompt = str(neg_prompt).strip()\n\n    if not pipeline_ready or stream_txt2img is None:\n        return\n\n    state = (current_prompt_a, current_prompt_b, current_prompt_blend, current_neg_prompt)\n    if state == applied_prompt_state:\n        return\n\n    try:\n        device = "cuda" if torch.cuda.is_available() else "cpu"\n        core_pipe = stream_txt2img.stream.pipe\n\n        def encode_cached(prompt):\n            key = (id(core_pipe), prompt, current_neg_prompt)\n            embedding = prompt_embedding_cache.pop(key, None)\n            if embedding is None:\n                encoded = core_pipe.encode_prompt(\n                    prompt=prompt,\n                    device=device,\n                    num_images_per_prompt=1,\n                    do_classifier_free_guidance=False,\n                    negative_prompt=current_neg_prompt,\n                )\n                embedding = (encoded[0], encoded[2]) if diffusion_model == \'sdxl-turbo\' else encoded[0]\n                # Bound retained GPU memory while editing prompts interactively.\n                if len(prompt_embedding_cache) >= 4:\n                    del prompt_embedding_cache[next(iter(prompt_embedding_cache))]\n            prompt_embedding_cache[key] = embedding\n            return embedding\n\n        with torch.no_grad():\n            embed_a = encode_cached(current_prompt_a)\n\n            if current_prompt_b and current_prompt_blend > 0.001:\n                embed_b = encode_cached(current_prompt_b)\n                alpha = current_prompt_blend\n                blended = (tuple((1.0 - alpha) * a + alpha * b for a, b in zip(embed_a, embed_b))\n                           if diffusion_model == \'sdxl-turbo\' else (1.0 - alpha) * embed_a + alpha * embed_b)\n            else:\n                blended = embed_a\n\n            # Apply blended embedding tensor to both pipelines\n            for s in [stream_txt2img, stream_img2img]:\n                if s is not None:\n                    target = s.stream if hasattr(s, "stream") else s\n                    if hasattr(target, "prompt_embeds"):\n                        tokens = blended[0] if diffusion_model == \'sdxl-turbo\' else blended\n                        target.prompt_embeds = tokens.repeat(target.batch_size, 1, 1)\n                        if diffusion_model == \'sdxl-turbo\':\n                            target.unet.set_conditioning(blended[1])\n\n        applied_prompt_state = state\n        logger.debug("Prompt embeddings updated (blend=%.2f)", current_prompt_blend)\n    except Exception as e:\n        logger.error(f"Error in prompt blending: {traceback.format_exc()}")\n        raise\n\ndef apply_strength(new_strength):\n    global current_strength, stream_img2img, applied_prompt_state\n    current_strength = max(0.1, min(1.0, float(new_strength)))\n    try:\n        if stream_img2img is not None:\n            if hasattr(stream_img2img, "update_strength"):\n                stream_img2img.update_strength(current_strength)\n            elif hasattr(stream_img2img, "stream") and hasattr(stream_img2img.stream, "t_list"):\n                t0 = max(5, min(44, int(45 - current_strength * 40)))\n                stream_img2img.stream.t_list = [t0, 45]\n                stream_img2img.prepare(prompt=current_prompt_a, num_inference_steps=50)\n                logger.info(f"Updated img2img denoise strength to {current_strength:.2f} (t_index_list={[t0, 45]})")\n            # prepare() replaces prompt_embeds; restore the cached blend afterward.\n            applied_prompt_state = None\n    except Exception as e:\n        logger.error(f"Error updating strength: {e}")\n        raise\n\ndef get_gpu_telemetry():\n    global telemetry_cache\n    # nvidia-smi can take tens of milliseconds. Sample at most once every 2s,\n    # and call this function off the asyncio event loop.\n    with telemetry_lock:\n        now = time.perf_counter()\n        if now - telemetry_cache[0] < 2.0:\n            return telemetry_cache[1:]\n        vram_str, temp_str = sample_gpu_telemetry()\n        telemetry_cache = (time.perf_counter(), vram_str, temp_str)\n        return vram_str, temp_str\n\n\ndef sample_gpu_telemetry():\n    vram_str = "--"\n    temp_str = "--"\n    if torch.cuda.is_available():\n        try:\n            allocated = torch.cuda.memory_allocated() / (1024**3)\n            total = torch.cuda.get_device_properties(0).total_memory / (1024**3)\n            pct = (allocated / total) * 100\n            vram_str = f"{allocated:.1f}/{total:.1f}GB ({pct:.0f}%)"\n        except Exception:\n            pass\n        try:\n            import subprocess\n            out = subprocess.check_output(\n                ["nvidia-smi", "--query-gpu=temperature.gpu", "--format=csv,noheader,nounits"],\n                timeout=1,\n                text=True\n            ).strip()\n            temp_str = f"{out}°C"\n        except Exception:\n            pass\n    return vram_str, temp_str\n\ndef run_model_inference(input_image=None, mode=None):\n    """\n    Runs StreamDiffusion inference under torch.no_grad() inside the executor thread.\n    Never use torch.inference_mode() because StreamDiffusion caches tensors between steps.\n    """\n    with torch.no_grad():\n        mode = mode or current_mode\n        if mode in ("img2img", "v2v") and input_image is not None and stream_img2img is not None:\n            return stream_img2img(input_image)\n        if stream_txt2img is not None:\n            return stream_txt2img()\n        return None\n\n\ndef process_frame(mode, frame_bytes=None, max_frame_bytes=None):\n    """Decode, infer, and encode on the worker; return complete frame timings."""\n    started = time.perf_counter()\n    if is_world_backend or is_krea_backend or is_ltx_backend:\n        if is_world_backend and frame_bytes is not None:\n            world_backend.set_seed_image(frame_bytes)\n        result = ltx_backend.pop_frame() if is_ltx_backend else stateful_backend().next_frame()\n        if result is None:\n            return None\n        out_bgr, batch_ms = result\n        jpg = compress_frame(out_bgr, max_frame_bytes, allow_oversize=True)\n        # The GPU readout is the generated chunk duration, not single-frame latency.\n        return jpg, batch_ms, (time.perf_counter() - started) * 1000\n    input_image = None\n    if frame_bytes is not None:\n        in_bgr = cv2.imdecode(np.frombuffer(frame_bytes, np.uint8), cv2.IMREAD_COLOR)\n        if in_bgr is None:\n            return None\n        input_image = Image.fromarray(cv2.cvtColor(in_bgr, cv2.COLOR_BGR2RGB)).resize((output_width, output_height))\n\n    inference_started = time.perf_counter()\n    output = run_model_inference(input_image, mode=mode)\n    inf_ms = (time.perf_counter() - inference_started) * 1000\n    if output is None:\n        return None\n    if isinstance(output, Image.Image):\n        out_bgr = cv2.cvtColor(np.asarray(output), cv2.COLOR_RGB2BGR)\n    elif torch.is_tensor(output):\n        arr = output.detach()\n        if arr.ndim == 4:\n            arr = arr[0]\n        arr = arr.cpu().permute(1, 2, 0).float().numpy()\n        out_bgr = cv2.cvtColor((np.clip(arr, 0, 1) * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)\n    else:\n        out_bgr = np.asarray(output)\n    jpg_bytes = compress_frame(out_bgr, max_bytes=max_frame_bytes,\n                               allow_oversize=max_frame_bytes is not None)\n    return jpg_bytes, inf_ms, (time.perf_counter() - started) * 1000\n\n\ndef apply_controls(data):\n    """Apply an entire update between inference calls on the same worker."""\n    global current_mode, current_seed, output_width, output_height, last_pipeline_error, pipeline_ready\n    global pipeline_revision, current_prompt_a\n    if is_ltx_backend:\n        data = validated_controls(data, (output_width, output_height))\n        reset = ltx_backend.configure(data)\n        current_seed, current_prompt_a = ltx_backend.seed, ltx_backend.prompt\n        if reset:\n            pipeline_revision += 1\n        return\n    if is_krea_backend:\n        data = validated_controls(data, (output_width, output_height))\n        previous_seed = krea_backend.seed\n        krea_backend.configure(data)\n        current_seed, current_prompt_a = krea_backend.seed, krea_backend.prompt\n        if data.get(\'krea_reset\') or previous_seed != current_seed:\n            pipeline_revision += 1\n        return\n    if is_world_backend:\n        data = validated_controls(data, (output_width, output_height))\n        world_backend.configure(data)\n        current_seed = world_backend.seed\n        if data.get(\'world_reset\'):\n            pipeline_revision += 1\n        return\n    data = validate_controls(data, (output_width, output_height))\n    width, height = requested_resolution(data)\n    if resolution_locked() and (width, height) != (output_width, output_height):\n        raise ValueError(\'TensorRT resolution is locked; select the startup resolution and restart the server\')\n    state_fields = (\'current_mode\', \'current_seed\', \'current_strength\', \'current_prompt_a\',\n                    \'current_prompt_b\', \'current_prompt_blend\', \'current_neg_prompt\',\n                    \'output_width\', \'output_height\')\n    previous = {name: globals()[name] for name in state_fields}\n    resizing = (width, height) != (output_width, output_height) or not pipeline_ready\n    try:\n        _apply_validated_controls(data, width, height, resizing)\n        last_pipeline_error = None\n    except Exception as exc:\n        globals().update(previous)\n        last_pipeline_error = "Configuration failed; see server logs"\n        # A failed constructor can retain tensors in its traceback. Release\n        # those frames before attempting to allocate the old model buffers.\n        exc.__traceback__ = None\n        try:\n            if resizing:\n                load_pipeline()\n            else:\n                apply_strength(current_strength)\n                set_seed(current_seed)\n                apply_prompts_and_blend()\n        except Exception:\n            pipeline_ready = False\n            logger.exception("Could not restore the previous pipeline configuration")\n        raise RuntimeError(f"Configuration failed: {exc}") from exc\n\n\ndef _apply_validated_controls(data, width, height, resizing):\n    global current_mode, current_seed, output_width, output_height\n    global current_prompt_a, current_prompt_b, current_prompt_blend, current_neg_prompt, current_strength\n    if resizing:\n        # Configure state before allocating fresh buffers; avoid preparing the\n        # old models only to discard them immediately afterward.\n        for key, name in ((\'prompt_a\', \'current_prompt_a\'), (\'prompt_b\', \'current_prompt_b\'),\n                          (\'blend\', \'current_prompt_blend\'), (\'neg_prompt\', \'current_neg_prompt\'),\n                          (\'strength\', \'current_strength\'), (\'seed\', \'current_seed\'), (\'mode\', \'current_mode\')):\n            if key in data:\n                globals()[name] = data[key]\n        output_width, output_height = width, height\n        load_pipeline()\n        return\n    mode = data.get("mode")\n    if mode is not None:\n        new_mode = mode\n        if new_mode != current_mode:\n            current_mode = new_mode\n            logger.info("Switching mode to: %s", current_mode)\n    seed = data.get("seed")\n    if seed is not None and int(seed) != current_seed:\n        current_seed = int(seed)\n        set_seed(current_seed)\n    strength = data.get("strength")\n    if strength is not None and abs(float(strength) - current_strength) > 0.01:\n        apply_strength(strength)\n    apply_prompts_and_blend(\n        data.get("prompt_a", data.get("prompt")), data.get("prompt_b"),\n        data.get("blend"), data.get("neg_prompt"),\n    )\n\ndef compress_frame(arr_bgr, max_bytes=None, allow_oversize=False):\n    """\n    Bound payload size without silently changing the selected output resolution.\n    """\n    if max_bytes is None:\n        # Give larger outputs a proportional byte budget, retaining the tested\n        # 48KB default for <=512px frames. TLS packets are still only 2KB each.\n        height, width = arr_bgr.shape[:2]\n        max_bytes = max(48000, round(48000 * width * height / (512 * 512)))\n    qualities = (60, 50, 40, 30, 20, 10, 5, 1)\n    key = (*arr_bgr.shape[:2], max_bytes)\n    hint, count = jpeg_quality_hints.pop(key, (0, 0))\n    # Start near the previous successful quality instead of encoding a large\n    # image up to eight times on every frame. Periodically probe one step up.\n    start = max(0, hint - 1) if count % 30 == 0 else hint\n    smallest = None\n    for index in range(start, len(qualities)):\n        ret, buf = cv2.imencode(\'.jpg\', arr_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), qualities[index],\n                                               int(cv2.IMWRITE_JPEG_OPTIMIZE), 1])\n        if ret and (smallest is None or len(buf) < len(smallest)):\n            smallest = buf\n        if ret and len(buf) <= max_bytes:\n            if len(jpeg_quality_hints) >= 32:\n                del jpeg_quality_hints[next(iter(jpeg_quality_hints))]\n            next_hint = max(0, index - 1) if len(buf) < max_bytes / 2 else index\n            jpeg_quality_hints[key] = (next_hint, count + 1)\n            return buf.tobytes()\n    # Some full-resolution noise/pattern images exceed a small target even at\n    # minimum quality. Keep streaming their pixels within the protocol limit.\n    if allow_oversize and smallest is not None and len(smallest) <= 255 * OUTPUT_CHUNK_BYTES:\n        if len(jpeg_quality_hints) >= 32:\n            del jpeg_quality_hints[next(iter(jpeg_quality_hints))]\n        jpeg_quality_hints[key] = (len(qualities) - 1, count + 1)\n        if count % 30 == 0:\n            logger.warning(\'JPEG at minimum quality exceeds the %d-byte target (%d bytes); \'\n                           \'preserving full output resolution\', max_bytes, len(smallest))\n        return smallest.tobytes()\n    raise ValueError("Could not encode an output JPEG within the frame byte budget")\n\n\ndef create_txt2img_pipeline():\n    if diffusion_model == \'sdxl-turbo\':\n        from sdxl_stream import SDXLStreamWrapper\n        return SDXLStreamWrapper.load(output_width, output_height)\n    # StreamDiffusion\'s bundled TensorRT/Polygraphy path targets old CUDA 11/12\n    # runtimes. Load the native models, then compile their UNets explicitly.\n    return StreamDiffusionWrapper(\n        model_id_or_path="stabilityai/sd-turbo",\n        t_index_list=[0], frame_buffer_size=1,\n        width=output_width, height=output_height, warmup=5,\n        acceleration="none", mode="txt2img",\n        use_lcm_lora=False, use_tiny_vae=True, cfg_type="none",\n        use_denoising_batch=True,\n        device="cuda" if torch.cuda.is_available() else "cpu", seed=current_seed,\n    )\n\n\ndef load_pipeline():\n    global stream_txt2img, stream_img2img, pipeline_ready, active_engine, applied_prompt_state\n    global pipeline_revision, img2img_engine, world_backend, krea_backend, ltx_backend\n    if is_ltx_backend:\n        if enable_tensorrt:\n            raise RuntimeError(\'LTX uses its own runtime; disable TensorRT\')\n        from pathlib import Path\n        from ltx_backend import LTXBackend\n        pipeline_ready = False\n        ltx_backend = LTXBackend.load(Path(__file__).resolve().parent)\n        pipeline_revision += 1\n        pipeline_ready = True\n        print(\'>>> LTX pipeline ready for Generate Clip; first use loads weights.\', flush=True)\n        return\n    if is_krea_backend:\n        if enable_tensorrt:\n            raise RuntimeError(\'Krea uses its own INT8 runtime; disable TensorRT\')\n        from pathlib import Path\n        from krea_backend import KreaBackend\n        pipeline_ready = False\n        krea_backend = KreaBackend.load(Path(__file__).resolve().parent,\n                                        clip_blocks=int(os.environ.get(\'KREA_CLIP_BLOCKS\', \'64\')))\n        pipeline_revision += 1\n        pipeline_ready = True\n        print(\'>>> Krea INT8 loaded. Submit Prompt A in TouchDesigner; first chunk may compile kernels.\', flush=True)\n        return\n    if is_world_backend:\n        if enable_tensorrt:\n            raise RuntimeError(\'Waypoint uses its own INT8/BF16 runtime; disable TensorRT\')\n        from waypoint_backend import WaypointBackend\n        pipeline_ready = False\n        world_backend = WaypointBackend.load(os.environ.get(\'WAYPOINT_QUANT\', \'intw8a8\'),\n                                            os.environ.get(\'WAYPOINT_COMPILE_PROFILE\', \'fast\'),\n                                            prompt_seeds=os.environ.get(\'WAYPOINT_PROMPT_SEEDS\', \'0\') == \'1\',\n                                            prompt_seed_model=os.environ.get(\'WAYPOINT_PROMPT_SEED_MODEL\', \'sd-turbo\'))\n        pipeline_revision += 1\n        pipeline_ready = True\n        print(\'>>> Waypoint loaded. Seed an image from TouchDesigner; first generation compiles kernels.\', flush=True)\n        return\n    if StreamDiffusionWrapper is None and diffusion_model != \'sdxl-turbo\':\n        raise RuntimeError("StreamDiffusion is not installed. Run Colab setup or workstation.py setup first.")\n    pipeline_revision += 1\n    prompt_embedding_cache.clear()\n    applied_prompt_state = None\n    pipeline_ready = False\n    # Release old model buffers before allocating a different resolution.\n    stream_img2img = None\n    stream_txt2img = None\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    active_engine = "PyTorch (FP16)"\n    img2img_engine = "PyTorch (FP16)"\n    requested = "TensorRT" if enable_tensorrt else "PyTorch FP16"\n    print(f\'>>> Initializing {model_label()} on {device.upper()}; requested backend: {requested}\', flush=True)\n    print(f\'>>> Initializing Text-to-Image pipeline (single-image {model_label()}, t_index_list=[0])...\', flush=True)\n    stream_txt2img = create_txt2img_pipeline()\n    set_seed(current_seed)\n\n    # 2. Image-to-Image / StreamV2V pipeline: initial t_index_list based on current_strength\n    t0_init = max(5, min(44, int(45 - current_strength * 40)))\n    print(f">>> Initializing Image-to-Image / StreamV2V pipeline (t_index_list=[{t0_init}, 45], strength={current_strength:.2f})...")\n    try:\n        from streamdiffusion import StreamDiffusion\n        core_pipe = stream_txt2img.stream.pipe\n        stream_core = StreamDiffusion(\n            pipe=core_pipe,\n            t_index_list=[t0_init, 45],\n            torch_dtype=core_pipe.dtype,\n            width=output_width,\n            height=output_height,\n            frame_buffer_size=1,\n            use_denoising_batch=True,\n            cfg_type="none",\n        )\n        if hasattr(stream_txt2img.stream, "vae") and stream_txt2img.stream.vae:\n            stream_core.vae = stream_txt2img.stream.vae\n\n        class SharedImg2ImgWrapper:\n            def __init__(self, core):\n                self.stream = core\n            def prepare(self, prompt, **kwargs):\n                self.stream.prepare(prompt, **kwargs)\n            def update_prompt(self, prompt):\n                self.stream.update_prompt(prompt)\n            def update_strength(self, strength: float):\n                # Dynamically maps strength [0.1..1.0] to start timestep index [44..5]\n                t0 = max(5, min(44, int(45 - strength * 40)))\n                new_t_list = [t0, 45]\n                if getattr(self.stream, "t_list", None) != new_t_list:\n                    self.stream.t_list = new_t_list\n                    self.stream.prepare(prompt=current_prompt_a, num_inference_steps=50)\n                    logger.info(f"Updated img2img denoise strength to {strength:.2f} (t_index_list={new_t_list})")\n            def __call__(self, image):\n                x_output = self.stream(image)\n                if diffusion_model == \'sdxl-turbo\':\n                    return self.stream.image_processor.postprocess(x_output, output_type=\'pil\')[0]\n                pil_out = postprocess_image(x_output, output_type="pil")\n                return pil_out[0] if isinstance(pil_out, list) else pil_out\n\n        stream_img2img = SharedImg2ImgWrapper(stream_core)\n    except Exception as e:\n        if diffusion_model == \'sdxl-turbo\':\n            raise RuntimeError(\'SDXL img2img initialization failed; no other model was substituted\') from e\n        print(">>> Fallback to standalone img2img wrapper:", e)\n        stream_img2img = StreamDiffusionWrapper(\n            model_id_or_path="stabilityai/sd-turbo",\n            t_index_list=[t0_init, 45],\n            frame_buffer_size=1,\n            width=output_width,\n            height=output_height,\n            warmup=5,\n            acceleration="none",\n            mode="img2img",\n            use_lcm_lora=False,\n            use_tiny_vae=True,\n            cfg_type="none",\n            use_denoising_batch=True,\n            device=device,\n            seed=current_seed,\n        )\n        img2img_engine = "PyTorch (FP16)"\n\n    print(">>> Preparing both pipelines with initial prompt...")\n    stream_txt2img.prepare(prompt=current_prompt_a, num_inference_steps=50)\n    stream_img2img.prepare(prompt=current_prompt_a, num_inference_steps=50)\n\n    if enable_tensorrt:\n        # Batch one serves txt2img; the two-timestep img2img stream uses batch\n        # two. Compile and validate both before reporting readiness.\n        txt_source, img_source = stream_txt2img.stream.unet, stream_img2img.stream.unet\n        is_xl = diffusion_model == \'sdxl-turbo\'\n        txt_unet = compile_tensorrt_unet(txt_source.model if is_xl else txt_source, output_width, output_height, 1)\n        img_unet = compile_tensorrt_unet(img_source.model if is_xl else img_source, output_width, output_height, 2)\n        stream_txt2img.stream.unet = txt_source.with_model(txt_unet) if is_xl else txt_unet\n        stream_img2img.stream.unet = img_source.with_model(img_unet) if is_xl else img_unet\n        active_engine = img2img_engine = "TensorRT (UNet)"\n        print(">>> TensorRT UNet acceleration enabled for BOTH generation modes; TinyVAE remains PyTorch.", flush=True)\n\n    print(">>> Warming up models...")\n    with torch.no_grad():\n        for _ in range(4):\n            stream_txt2img()\n        dummy_img = Image.new("RGB", (output_width, output_height), color=(128, 128, 128))\n        for _ in range(3):\n            stream_img2img(dummy_img)\n\n    pipeline_ready = True\n    apply_prompts_and_blend()\n    print(">>> StreamDiffusion pipeline is READY! <<<", flush=True)\n\n@app.on_event("startup")\nasync def startup_event():\n    loop = asyncio.get_event_loop()\n    await loop.run_in_executor(model_executor, load_pipeline)\n\n\n@app.on_event("shutdown")\nasync def close_ltx_runtime():\n    if is_ltx_backend and ltx_backend is not None:\n        await asyncio.get_running_loop().run_in_executor(model_executor, ltx_backend.close)\n\n\n@app.get("/")\ndef index():\n    return HTMLResponse("""\n    <html>\n        <head><title>StreamDiffusion TouchDesigner Server</title></head>\n        <body style="font-family: sans-serif; padding: 40px; background: #121212; color: #fff;">\n            <h2>StreamDiffusion (SD-Turbo) Bridge Server</h2>\n            <p>Status: <span style="color: #0f0;">Running</span></p>\n            <p>Connect your TouchDesigner client via the WebSocket endpoint: <code>/ws/stream</code></p>\n        </body>\n    </html>\n    """)\n\n\n@app.get("/health")\ndef health():\n    return JSONResponse({\n        "ready": pipeline_ready, "protocol": PROTOCOL_VERSION,\n        "mode": current_mode, "engine": engine_for_mode(),\n        "model": model_label(), "diffusion_model": diffusion_model,\n        "width": output_width, "height": output_height,\n        "client_connected": active_client is not None,\n        "last_error": last_pipeline_error,\n        "requested_engine": engine_for_mode() if is_world_backend or is_krea_backend or is_ltx_backend else "TensorRT" if enable_tensorrt else "PyTorch (FP16)",\n        "backend": backend_name,\n        "session_id": os.environ.get(\'BRIDGE_SESSION_ID\'),\n        "authentication_required": bool(access_token),\n        "resolution_locked": resolution_locked(),\n        **world_progress(),\n    }, status_code=200 if pipeline_ready else 503)\n\n\ndef configuration_error(code):\n    """Public status contains no internal exception text or stack information."""\n    if code == \'resolution_locked\':\n        status = f\'Output fixed at {output_width} x {output_height}; select the matching backend and startup size\'\n    elif code == "invalid_control":\n        status = "Invalid controls: check backend, output size, and control values"\n    elif is_ltx_backend:\n        status = \'LTX configuration failed; inspect server logs, then Generate Clip to retry\'\n    elif is_krea_backend:\n        status = \'Krea configuration failed; see server logs, then reset the clip\'\n    elif is_world_backend:\n        status = \'World configuration failed; see server logs, then reset or reseed the world\'\n    elif pipeline_ready:\n        status = "Configuration failed; previous settings restored. See server logs for details"\n    else:\n        status = "Configuration failed; see server logs, then retry or restart the server"\n    return {"type": "error", "code": code, "status": status,\n            "ready": pipeline_ready, "mode": current_mode,\n            "width": output_width, "height": output_height, \'engine\': engine_for_mode(),\n            \'resolution_locked\': resolution_locked()}\n\nasync def authorize_websocket(websocket):\n    # Authenticate before reserving the single controller or exposing frames.\n    # Tokens travel in a message, never a URL or a request log.\n    await websocket.accept()\n    if access_token:\n        await websocket.send_text(json.dumps({\'type\': \'auth_required\'}))\n        try:\n            message = await asyncio.wait_for(websocket.receive(), timeout=10)\n            if message.get(\'type\') == \'websocket.disconnect\':\n                return False\n            data = json.loads(message.get(\'text\', \'\'))\n            authenticated = (isinstance(data, dict) and data.get(\'type\') == \'auth\'\n                             and isinstance(data.get(\'token\'), str)\n                             and hmac.compare_digest(data[\'token\'].encode(), access_token.encode()))\n        except (asyncio.TimeoutError, ValueError, TypeError, WebSocketDisconnect):\n            authenticated = False\n        if not authenticated:\n            await websocket.send_text(json.dumps({\'type\': \'error\', \'code\': \'unauthorized\',\n                                                 \'status\': \'Access token missing or incorrect\'}))\n            await websocket.close(code=1008)\n            return False\n    return True\n\n\n@app.websocket(\'/ws/ping\')\nasync def websocket_ping_endpoint(websocket: WebSocket):\n    if not await authorize_websocket(websocket):\n        return\n    try:\n        await websocket.send_text(json.dumps({\'type\': \'probe_ready\'}))\n        while True:\n            message = await websocket.receive()\n            if message.get(\'type\') == \'websocket.disconnect\':\n                break\n            if message.get(\'text\') and len(message[\'text\']) <= 1024:\n                try:\n                    data = json.loads(message[\'text\'])\n                    if isinstance(data, dict) and data.get(\'type\') == \'ping\':\n                        await websocket.send_text(json.dumps({\'type\': \'pong\', \'sent_at\': data.get(\'sent_at\')}))\n                except (TypeError, ValueError):\n                    pass\n    except WebSocketDisconnect:\n        pass\n\n\n@app.websocket("/ws/stream")\nasync def websocket_stream_endpoint(websocket: WebSocket):\n    global active_client\n    if not await authorize_websocket(websocket):\n        return\n    # Both modes share one mutable model; an additional controller must not\n    # silently overwrite the active performer\'s prompts and resolution.\n    if active_client is not None:\n        await websocket.send_text(json.dumps({"type": "error", "code": "server_busy",\n                                             "status": "Server already has an active TouchDesigner client"}))\n        await websocket.close(code=1013)\n        return\n    active_client = websocket\n    logger.info("TouchDesigner client connected to WebSocket.")\n    loop = asyncio.get_running_loop()\n    send_lock = asyncio.Lock()\n    frame_lock = asyncio.Lock()\n    frame_credit = asyncio.Event()\n    frame_credit.set()\n    ack_enabled, frame_window = False, 3\n    video_quality = \'balanced\'\n    outstanding_frames = {}\n    stop_event = asyncio.Event()\n    frame_available = asyncio.Event()\n    control_available = asyncio.Event()\n    pending_frame = None\n    pending_control = {}\n    assembler = FrameAssembler(timeout=15.0 if is_world_backend else 1.0)\n    frame_counter = 0\n    latest_stats = {}\n    world_seed_until = 0.0\n\n    async def send_json(data):\n        async with send_lock:\n            await websocket.send_text(json.dumps(data))\n\n    async def wait_for_frame_credit():\n        started = time.perf_counter()\n        while ack_enabled and len(outstanding_frames) >= frame_window:\n            frame_credit.clear()\n            await frame_credit.wait()\n        return (time.perf_counter() - started) * 1000\n\n    async def send_chunked_frame(jpg_bytes):\n        nonlocal frame_counter\n        chunk_size = OUTPUT_CHUNK_BYTES\n        total_chunks = (len(jpg_bytes) + chunk_size - 1) // chunk_size\n        if not 1 <= total_chunks <= 255:\n            raise ValueError("Output frame exceeds the binary protocol chunk limit")\n        async with frame_lock:\n            credit_ms = await wait_for_frame_credit()\n            frame_counter = (frame_counter + 1) % 256\n            started = time.perf_counter()\n            if ack_enabled:\n                outstanding_frames[frame_counter] = started\n            for chunk_idx in range(total_chunks):\n                chunk = jpg_bytes[chunk_idx * chunk_size:(chunk_idx + 1) * chunk_size]\n                async with send_lock:\n                    await websocket.send_bytes(bytes([frame_counter, chunk_idx, total_chunks]) + chunk)\n                if chunk_idx % 4 == 3:\n                    await asyncio.sleep(0)\n            return (time.perf_counter() - started) * 1000, credit_ms\n\n    async def telemetry_loop():\n        next_log = time.perf_counter() + 10\n        while not stop_event.is_set():\n            await asyncio.sleep(1.0)\n            vram, temp = await loop.run_in_executor(None, get_gpu_telemetry)\n            progress = world_progress()\n            if latest_stats or progress:\n                await send_json({\n                    "type": "stats", "engine": engine_for_mode(), "model": model_label(),\n                    "vram": vram, "temp": temp, **latest_stats,\n                    \'ready\': pipeline_ready, \'inflight_frames\': len(outstanding_frames),\n                    **progress,\n                })\n                if time.perf_counter() >= next_log:\n                    if progress and progress[\'world_phase\'] != \'streaming\':\n                        logger.info(\'%s progress: %s\', backend_name, progress[\'world_status\'])\n                    logger.info(\'delivery: %s, %s bytes/frame, send=%sms, display ACK RTT=%sms, \'\n                                \'buffer wait=%sms, client cook=%sms, window=%d, inflight=%d\',\n                                video_quality, latest_stats.get(\'frame_bytes\', \'--\'),\n                                latest_stats.get(\'send_ms\', \'--\'), latest_stats.get(\'delivery_ms\', \'--\'),\n                                latest_stats.get(\'credit_wait_ms\', \'--\'), latest_stats.get(\'display_ms\', \'--\'),\n                                frame_window, len(outstanding_frames))\n                    next_log = time.perf_counter() + 10\n\n    async def control_loop():\n        nonlocal pending_control, pending_frame\n        while not stop_event.is_set():\n            await control_available.wait()\n            control_available.clear()\n            data, pending_control = pending_control, {}\n            try:\n                data = validated_controls(data, (output_width, output_height))\n                width, height = requested_resolution(data)\n                if resolution_locked() and (width, height) != (output_width, output_height):\n                    await send_json(configuration_error(\'resolution_locked\'))\n                    continue\n                resizing = (width, height) != (output_width, output_height) or not pipeline_ready\n                if resizing:\n                    pending_frame = None\n                    frame_available.clear()\n                    assembler.clear()\n                    backend = \'TensorRT\' if enable_tensorrt else \'PyTorch\'\n                    await send_json({\'type\': \'status\', \'status\': f\'Loading {width} x {height} {backend} output...\',\n                                     \'ready\': False, \'engine\': f\'{backend} (Preparing)\'})\n                if is_ltx_backend:\n                    # LTX configuration only mutates revisioned CPU state.\n                    # Prompt edits/pause must not queue behind a whole GPU clip.\n                    apply_controls(data)\n                else:\n                    await loop.run_in_executor(model_executor, apply_controls, data)\n                await send_json({"type": "configured", "status": "Connected (Streaming)",\n                                 "width": output_width, "height": output_height, "engine": engine_for_mode(),\n                                 "ready": pipeline_ready, "mode": current_mode,\n                                 \'resolution_locked\': resolution_locked(), **world_progress()})\n                if current_mode not in (\'img2img\', \'world\'):\n                    pending_frame = None\n                    frame_available.clear()\n                    assembler.clear()\n            except ValueError:\n                logger.exception("Invalid control message")\n                await send_json(configuration_error("invalid_control"))\n            except Exception:\n                logger.exception("Error applying control message")\n                await send_json(configuration_error("configuration_failed"))\n\n    async def generation_loop(mode):\n        nonlocal pending_frame\n        frame_count = 0\n        while not stop_event.is_set():\n            try:\n                frame_bytes = None\n                if mode == "img2img":\n                    await frame_available.wait()\n                    frame_available.clear()\n                    frame_bytes, pending_frame = pending_frame, None\n                    if frame_bytes is None:\n                        continue\n                elif not pipeline_ready or current_mode != mode:\n                    await asyncio.sleep(0.01)\n                    continue\n\n                if mode in (\'world\', \'krea\', \'ltx\'):\n                    # Stateful generation requires bounded delivery and a configured controller.\n                    if not ack_enabled:\n                        await asyncio.sleep(0.02)\n                        continue\n                    if mode == \'world\' and pending_frame is not None:\n                        frame_bytes, pending_frame = pending_frame, None\n                    elif not (ltx_backend.can_deliver if mode == \'ltx\' else stateful_backend().can_generate):\n                        await asyncio.sleep(0.02)\n                        continue\n\n                if not pipeline_ready or current_mode != mode:\n                    continue\n                credit_ms = await wait_for_frame_credit()\n                if not pipeline_ready or current_mode != mode:\n                    continue\n                if mode == \'img2img\' and pending_frame is not None:\n                    frame_bytes, pending_frame = pending_frame, None\n                    frame_available.clear()\n                started = time.perf_counter()\n                revision = pipeline_revision\n                frame_quality = video_quality\n                budget = video_byte_budget(output_width, output_height, frame_quality)\n                # LTX pop/JPEG work is CPU-only and can overlap the single GPU\n                # worker rendering one lookahead clip. Other backends stay serial.\n                executor = None if mode == \'ltx\' else model_executor\n                result = await loop.run_in_executor(executor, process_frame, mode, frame_bytes, budget)\n                # Discard a completed frame if its mode was switched during inference.\n                if result is not None and current_mode == mode and revision == pipeline_revision:\n                    jpg_bytes, inf_ms, process_ms = result\n                    send_ms, extra_credit_ms = await send_chunked_frame(jpg_bytes)\n                    frame_count += 1\n                    latest_stats.update(inf_ms=round(inf_ms, 1), process_ms=round(process_ms, 1),\n                                        mode=mode, strength=current_strength, frame_bytes=len(jpg_bytes),\n                                        video_quality=frame_quality, send_ms=round(send_ms, 1),\n                                        credit_wait_ms=round(credit_ms + extra_credit_ms, 1))\n                    if frame_count % 100 == 1:\n                        logger.info("%s: frame #%d (inference %.1fms, processing %.1fms, %d bytes)",\n                                    mode, frame_count, inf_ms, process_ms, len(jpg_bytes))\n                if mode in (\'txt2img\', \'world\', \'krea\', \'ltx\'):\n                    fps = min(max_stream_fps, ltx_backend.playback_fps) if mode == \'ltx\' else max_stream_fps\n                    await asyncio.sleep(max(0.0, 1.0 / fps - (time.perf_counter() - started)))\n            except (WebSocketDisconnect, RuntimeError, OSError):\n                logger.exception("Streaming stopped in %s", mode)\n                stop_event.set()\n                # Wake receive() too, rather than leaving an idle connection alive.\n                await websocket.close()\n                return\n            except Exception:\n                logger.exception("Frame processing failed in %s", mode)\n                await asyncio.sleep(0.05)\n\n    async def ltx_render_loop():\n        while not stop_event.is_set():\n            if ack_enabled and pipeline_ready and current_mode == \'ltx\' and ltx_backend.needs_render:\n                try:\n                    await loop.run_in_executor(model_executor, ltx_backend.render_next_clip)\n                except ValueError:\n                    logger.exception(\'LTX render failed; waiting for an explicit retry\')\n            else:\n                await asyncio.sleep(0.02)\n\n    async def supervise(coroutine):\n        try:\n            await coroutine\n        except asyncio.CancelledError:\n            raise\n        except Exception:\n            logger.exception("Connection worker failed")\n            stop_event.set()\n            try:\n                await websocket.close()\n            except Exception:\n                pass\n\n    tasks = []\n    try:\n        if is_world_backend or is_krea_backend or is_ltx_backend:\n            # Queue behind any last GPU call from the previous controller.\n            await loop.run_in_executor(model_executor, stateful_backend().release_controls)\n        vram, temp = await loop.run_in_executor(None, get_gpu_telemetry)\n        await send_json({\n            "type": "welcome", "message": "StreamDiffusion server connected",\n            "engine": engine_for_mode(), "prompt_a": current_prompt_a,\n            "model": model_label(), "diffusion_model": diffusion_model,\n            "prompt_b": current_prompt_b, "blend": current_prompt_blend,\n            "mode": current_mode, "seed": current_seed, "strength": current_strength,\n            "width": output_width, "height": output_height,\n            "protocol": PROTOCOL_VERSION, "ready": pipeline_ready,\n            "vram": vram, "temp": temp,\n            \'resolution_locked\': resolution_locked(), \'latency_probe\': True, \'frame_ack\': True,\n            \'video_quality_profiles\': list(VIDEO_QUALITY_LIMITS),\n            \'backend\': backend_name,\n            \'supported_modes\': [\'ltx\'] if is_ltx_backend else [\'krea\'] if is_krea_backend else [\'world\'] if is_world_backend else [\'txt2img\', \'img2img\'],\n            \'world_seed_required\': is_world_backend and world_backend.seed_image is None,\n            \'inference_batch_frames\': ltx_backend.num_frames if is_ltx_backend else 12 if is_krea_backend else 4 if is_world_backend else 1,\n            **world_progress(),\n        })\n        modes = (\'ltx\',) if is_ltx_backend else (\'krea\',) if is_krea_backend else (\'world\',) if is_world_backend else (\'txt2img\', \'img2img\')\n        tasks = [asyncio.create_task(supervise(generation_loop(mode))) for mode in modes]\n        if is_ltx_backend:\n            tasks.append(asyncio.create_task(supervise(ltx_render_loop())))\n        tasks.append(asyncio.create_task(supervise(telemetry_loop())))\n        tasks.append(asyncio.create_task(supervise(control_loop())))\n        while not stop_event.is_set():\n            message = await websocket.receive()\n            if message.get("type") == "websocket.disconnect":\n                break\n            if message.get("text") is not None:\n                try:\n                    data = json.loads(message["text"])\n                    if not isinstance(data, dict):\n                        continue\n                    if data.get("type") == "ping":\n                        await send_json({"type": "pong", "sent_at": data.get("sent_at")})\n                    elif data.get(\'type\') == \'world_seed\' and is_world_backend:\n                        assembler.clear()\n                        world_seed_until = time.monotonic() + 15\n                        logger.info(\'Waypoint seed upload requested\')\n                    elif data.get(\'type\') == \'client_options\':\n                        quality = data.get(\'video_quality\', video_quality)\n                        if quality not in VIDEO_QUALITY_LIMITS:\n                            raise ValueError(\'Invalid video quality\')\n                        value = data.get(\'frame_window\', 3)\n                        if data.get(\'frame_ack\') is True and (not isinstance(value, int)\n                                or isinstance(value, bool) or not 1 <= value <= 6):\n                            raise ValueError(\'Invalid frame window\')\n                        video_quality = quality\n                        if data.get(\'frame_ack\') is True:\n                            ack_enabled, frame_window = True, value\n                            frame_credit.set()\n                    elif data.get(\'type\') == \'frame_ack\':\n                        frame_id = data.get(\'frame_id\')\n                        if isinstance(frame_id, int) and not isinstance(frame_id, bool):\n                            sent = outstanding_frames.pop(frame_id, None)\n                            if sent is not None:\n                                frame_credit.set()\n                                latest_stats[\'delivery_ms\'] = round((time.perf_counter() - sent) * 1000, 1)\n                                for name in (\'display_ms\', \'client_frame_ms\'):\n                                    value = data.get(name)\n                                    if (isinstance(value, (int, float)) and not isinstance(value, bool)\n                                            and 0 <= value <= 60000 and np.isfinite(value)):\n                                        latest_stats[name] = round(value, 1)\n                    elif data.get("type") == "control" or any(\n                        key in data for key in ("prompt", "prompt_a", "mode", "seed", "strength", "blend")\n                    ):\n                        # Merge rapid slider/audio changes while the worker is\n                        # busy; the latest value of each control wins.\n                        world_pulses = {key: True for key in (\'world_reset\', \'world_reseed\', \'krea_reset\', \'ltx_generate\')\n                                        if pending_control.get(key) is True}\n                        pending_control.update(data)\n                        if is_world_backend or is_krea_backend or is_ltx_backend:\n                            pending_control.update(world_pulses)\n                        control_available.set()\n                except (ValueError, TypeError):\n                    await send_json({"type": "error", "code": "invalid_message",\n                                     "status": "Send a valid JSON control object"})\n            elif (message.get("bytes") is not None and pipeline_ready\n                  and (current_mode == \'img2img\' or (is_world_backend and time.monotonic() < world_seed_until))):\n                raw = message["bytes"]\n                try:\n                    frame_bytes = assembler.feed(raw)\n                except ValueError as exc:\n                    logger.debug("Dropped invalid input frame: %s", exc)\n                    continue\n                if frame_bytes:\n                    world_seed_until = 0.0\n                    if is_world_backend:\n                        logger.info(\'Waypoint seed upload received: %d bytes\', len(frame_bytes))\n                        await send_json({\'type\': \'world_progress\', \'world_phase\': \'seeding\',\n                                         \'world_status\': \'World seed received; preparing seed and kernels\'})\n                    # Only one pending input: replace stale watchdog/recovery frames.\n                    # Receiving continues while the worker is busy with inference.\n                    pending_frame = frame_bytes\n                    frame_available.set()\n    except WebSocketDisconnect:\n        logger.info("TouchDesigner client disconnected.")\n    except Exception:\n        logger.exception("WebSocket error")\n    finally:\n        stop_event.set()\n        if is_ltx_backend:\n            # Invalidate a render immediately; its worker completion cannot refill\n            # queues or restore an old scene after this controller disconnects.\n            ltx_backend.release_controls()\n        for task in tasks:\n            task.cancel()\n        backend_to_release = stateful_backend() if is_world_backend or is_krea_backend or is_ltx_backend else None\n        def release_controller():\n            global active_client\n            try:\n                if backend_to_release is not None:\n                    backend_to_release.release_controls()\n            finally:\n                if active_client is websocket:\n                    active_client = None\n        # Submit before any cancellable await. An ASGI cancellation must not\n        # cancel this worker cleanup or leave its controller slot reserved.\n        # The worker serializes it behind already-running model operations.\n        cleanup = model_executor.submit(release_controller)\n        try:\n            await asyncio.gather(*tasks, return_exceptions=True)\n        finally:\n            await asyncio.shield(asyncio.wrap_future(cleanup))\n\n\nif __name__ == "__main__":\n    import uvicorn\n    # JPEG is already compressed; per-message deflate only adds CPU work.\n    uvicorn.run(app, host=os.environ.get(\'BRIDGE_HOST\', \'0.0.0.0\'),\n                port=int(os.environ.get(\'BRIDGE_PORT\', \'8000\')), log_level="info", ws_per_message_deflate=False,\n                ws_max_size=MAX_INPUT_FRAME_BYTES + 7)\n', 'bridge_protocol.py': '"""Shared presets, control validation, and bounded input frame reassembly."""\nimport math\nimport time\n\nPROTOCOL_VERSION = 1\nUPLOAD_CHUNK_BYTES = 4096\nOUTPUT_CHUNK_BYTES = 2048\nMAX_INPUT_FRAME_BYTES = 512 * 1024\nOUTPUT_PRESETS = (\n    (\'256\', \'256 x 256 (1:1 Fast)\', 256, 256),\n    (\'384\', \'384 x 384 (1:1)\', 384, 384),\n    (\'512\', \'512 x 512 (1:1 Default)\', 512, 512),\n    (\'768\', \'768 x 768 (1:1)\', 768, 768),\n    (\'512x384\', \'512 x 384 (4:3)\', 512, 384),\n    (\'768x576\', \'768 x 576 (4:3)\', 768, 576),\n    (\'1024x768\', \'1024 x 768 (4:3 Max)\', 1024, 768),\n    (\'384x512\', \'384 x 512 (3:4)\', 384, 512),\n    (\'576x768\', \'576 x 768 (3:4)\', 576, 768),\n    (\'768x1024\', \'768 x 1024 (3:4 Max)\', 768, 1024),\n    (\'512x288\', \'512 x 288 (16:9)\', 512, 288),\n    (\'768x432\', \'768 x 432 (16:9)\', 768, 432),\n    (\'1024x576\', \'1024 x 576 (16:9)\', 1024, 576),\n    (\'288x512\', \'288 x 512 (9:16)\', 288, 512),\n    (\'432x768\', \'432 x 768 (9:16)\', 432, 768),\n    (\'576x1024\', \'576 x 1024 (9:16)\', 576, 1024),\n)\nOUTPUT_RESOLUTIONS = tuple((width, height) for _, _, width, height in OUTPUT_PRESETS)\nVIDEO_QUALITY_LIMITS = {\'fast\': 48000, \'balanced\': 64000, \'detail\': None}\n\n\ndef video_byte_budget(width, height, quality=\'balanced\'):\n    """Select transport compression independently of generation dimensions."""\n    if quality not in VIDEO_QUALITY_LIMITS:\n        raise ValueError(\'Choose fast, balanced, or detail video quality\')\n    scaled = max(48000, round(48000 * width * height / (512 * 512)))\n    limit = VIDEO_QUALITY_LIMITS[quality]\n    return min(scaled, limit) if limit is not None else scaled\n\n\ndef integer(value, name):\n    if isinstance(value, bool):\n        raise ValueError(f\'{name} must be an integer\')\n    try:\n        number = int(value)\n    except (ValueError, TypeError, OverflowError) as exc:\n        raise ValueError(f\'{name} must be an integer\') from exc\n    if isinstance(value, float) and value != number:\n        raise ValueError(f\'{name} must be an integer\')\n    return number\n\n\ndef resolution(data, current=(512, 512)):\n    size = (integer(data.get(\'width\', current[0]), \'width\'),\n            integer(data.get(\'height\', current[1]), \'height\'))\n    if size not in OUTPUT_RESOLUTIONS:\n        raise ValueError(\'Choose a supported output preset, up to 1024 x 768 or 768 x 1024\')\n    return size\n\n\ndef validate_controls(data, current=(512, 512)):\n    """Validate the whole patch before mutating any model or application state."""\n    if not isinstance(data, dict):\n        raise ValueError(\'Control message must be a JSON object\')\n    clean = {}\n    if \'width\' in data or \'height\' in data:\n        clean[\'width\'], clean[\'height\'] = resolution(data, current)\n    for name, lower, upper in ((\'strength\', 0.1, 1.0), (\'blend\', 0.0, 1.0)):\n        if name in data:\n            try:\n                value = float(data[name])\n            except (ValueError, TypeError, OverflowError) as exc:\n                raise ValueError(f\'{name} must be a finite number\') from exc\n            if isinstance(data[name], bool) or not math.isfinite(value) or not lower <= value <= upper:\n                raise ValueError(f\'{name} must be between {lower} and {upper}\')\n            clean[name] = value\n    if \'seed\' in data:\n        seed = integer(data[\'seed\'], \'seed\')\n        if not 0 <= seed <= 4294967295:\n            raise ValueError(\'seed must be between 0 and 4294967295\')\n        clean[\'seed\'] = seed\n    if \'mode\' in data:\n        mode = str(data[\'mode\']).lower()\n        aliases = {\'txt2img\': \'txt2img\', \'text\': \'txt2img\', \'img2img\': \'img2img\', \'v2v\': \'img2img\'}\n        if mode not in aliases:\n            raise ValueError(\'mode must be txt2img or img2img\')\n        clean[\'mode\'] = aliases[mode]\n    if \'prompt\' in data and \'prompt_a\' not in data:\n        data = {**data, \'prompt_a\': data[\'prompt\']}\n    for name in (\'prompt_a\', \'prompt_b\', \'neg_prompt\'):\n        if name in data:\n            value = data[name]\n            if not isinstance(value, str) or len(value) > 4096:\n                raise ValueError(f\'{name} must be text of at most 4096 characters\')\n            if name == \'prompt_a\' and not value.strip():\n                raise ValueError(\'Prompt A cannot be empty\')\n            clean[name] = value.strip()\n    return clean\n\n\nclass FrameAssembler:\n    """Keep one incomplete frame, discarding stale or inconsistent chunks."""\n    def __init__(self, max_bytes=MAX_INPUT_FRAME_BYTES, timeout=1.0):\n        self.max_bytes = max_bytes\n        self.timeout = timeout\n        self.completed_id = None\n        self.clear()\n\n    def clear(self):\n        self.frame_id = None\n        self.chunks = []\n        self.size = 0\n        self.started = 0.0\n\n    def feed(self, raw, now=None):\n        now = time.monotonic() if now is None else now\n        if not raw.startswith(b\'FRM:\'):\n            if not raw.startswith(b\'\\xff\\xd8\') or len(raw) > self.max_bytes:\n                raise ValueError(\'Expected a bounded JPEG input frame\')\n            self.clear()\n            return raw\n        if len(raw) < 8 or len(raw) > UPLOAD_CHUNK_BYTES + 7:\n            self.clear()\n            raise ValueError(\'Invalid input chunk size\')\n        frame_id, index, count = raw[4:7]\n        if not count or index >= count:\n            self.clear()\n            raise ValueError(\'Invalid input chunk index or count\')\n        if frame_id == self.completed_id:\n            return None\n        if self.frame_id != frame_id or now - self.started > self.timeout:\n            self.clear()\n            self.frame_id = frame_id\n            self.chunks = [None] * count\n            self.started = now\n        if len(self.chunks) != count:\n            self.clear()\n            raise ValueError(\'Input chunk count changed within a frame\')\n        previous = self.chunks[index]\n        self.size += len(raw) - 7 - (len(previous) if previous is not None else 0)\n        if self.size > self.max_bytes:\n            self.clear()\n            raise ValueError(\'Input frame exceeds the byte limit\')\n        self.chunks[index] = raw[7:]\n        if all(chunk is not None for chunk in self.chunks):\n            result = b\'\'.join(self.chunks)\n            self.completed_id = frame_id\n            self.clear()\n            return result\n        return None\n', 'colab_launcher.py': '"""Own and monitor a bridge server and optional tunnel on Colab or a workstation."""\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport socket\nimport subprocess\nimport sys\nimport time\nimport urllib.error\nimport urllib.request\nfrom bridge_protocol import resolution\n\nTUNNEL_PROTOCOLS = (\'auto\', \'http2\', \'quic\')\n\n\nclass BridgeSession:\n    def __init__(self, tensorrt=False, max_fps=60, root=None, startup_timeout=None, tunnel_timeout=90,\n                 host=\'0.0.0.0\', port=8000, tunnel=True, advertise_host=None,\n                 server_script=\'server.py\', python_executable=None, cloudflared=None, access_token=None,\n                 output_size=(512, 512), allow_tensorrt_resize=False, engine_cache=None, reuse_engines=True,\n                 tunnel_protocol=\'auto\', backend=\'sd-turbo\', server_environment=None, diffusion_model=\'sd-turbo\'):\n        if backend not in (\'sd-turbo\', \'waypoint\', \'krea\', \'ltx\'):\n            raise ValueError(\'Backend must be sd-turbo, waypoint, krea, or ltx\')\n        if backend != \'sd-turbo\' and tensorrt:\n            raise ValueError(\'This backend uses its own runtime, not the SD-Turbo TensorRT engines\')\n        self.backend = backend\n        if diffusion_model not in (\'sd-turbo\', \'sdxl-turbo\'):\n            raise ValueError(\'Diffusion model must be sd-turbo or sdxl-turbo\')\n        if backend != \'sd-turbo\' and diffusion_model != \'sd-turbo\':\n            raise ValueError(\'Diffusion model selection applies only to the StreamDiffusion backend\')\n        self.diffusion_model = diffusion_model\n        self._expected_prompt_seed_model = None\n        # Per-launch options belong to the child, not the caller\'s process.\n        # None removes an inherited variable (e.g. a legacy model-cache alias).\n        self.server_environment = dict(server_environment or {})\n        if any(not isinstance(key, str) or not key or \'=\' in key or \'\\0\' in key\n               or (value is not None and (not isinstance(value, str) or \'\\0\' in value))\n               for key, value in self.server_environment.items()):\n            raise ValueError(\'Server environment must map valid names to strings or None\')\n        if not math.isfinite(float(max_fps)) or not 1 <= float(max_fps) <= 120:\n            raise ValueError(\'Maximum FPS must be between 1 and 120\')\n        self.root = Path(root or Path.cwd()).resolve()\n        if not 1 <= int(port) <= 65535:\n            raise ValueError(\'Port must be between 1 and 65535\')\n        self.host, self.port = host, int(port)\n        self.tunnel = bool(tunnel)\n        if tunnel_protocol not in TUNNEL_PROTOCOLS:\n            raise ValueError(\'Tunnel protocol must be auto, http2, or quic\')\n        self.tunnel_protocol = tunnel_protocol\n        self.advertise_host = advertise_host or (\'127.0.0.1\' if host == \'0.0.0.0\' else \'::1\' if host == \'::\' else host)\n        self.health_host = \'127.0.0.1\' if host == \'0.0.0.0\' else \'::1\' if host == \'::\' else host\n        self.server_script = (self.root / server_script).resolve()\n        self.python_executable = str(python_executable or sys.executable)\n        self.cloudflared = str(cloudflared or self.root / (\'cloudflared.exe\' if os.name == \'nt\' else \'cloudflared\'))\n        self.access_token = access_token\n        self.session_id = os.urandom(16).hex()\n        if backend == \'ltx\':\n            from ltx_backend import SIZES\n            if tuple(output_size) not in SIZES:\n                raise ValueError(\'Choose an LTX startup resolution preset\')\n        self.output_size = ((1280, 720) if backend == \'waypoint\' else (832, 480) if backend == \'krea\'\n                            else tuple(output_size) if backend == \'ltx\'\n                            else resolution({\'width\': output_size[0], \'height\': output_size[1]}))\n        self.allow_tensorrt_resize = bool(allow_tensorrt_resize)\n        self.engine_cache = Path(engine_cache or self.root / \'engines/torch-tensorrt\').resolve()\n        self.reuse_engines = bool(reuse_engines)\n        self.tensorrt = bool(tensorrt)\n        self.max_fps = float(max_fps)\n        self.startup_timeout = startup_timeout or (1800 if tensorrt else 600)\n        self.tunnel_timeout = tunnel_timeout\n        self.processes = {}\n        self.readers = {}\n        self.writers = {}\n        self.tunnel_url = None\n        self.tunnel_tail = \'\'\n\n    def endpoint(self, host=None, scheme=\'http\', path=\'\'):\n        host = host or self.health_host\n        host = f\'[{host}]\' if \':\' in host and not host.startswith(\'[\') else host\n        return f\'{scheme}://{host}:{self.port}{path}\'\n\n    def _spawn(self, name, command, env=None):\n        logs = self.root / \'bridge_logs\'\n        logs.mkdir(exist_ok=True)\n        path = logs / (name + \'.log\')\n        writer = path.open(\'w\', encoding=\'utf-8\')\n        self.writers[name] = writer\n        # Child processes write to regular files, so a quiet server or a full\n        # unread pipe cannot stall notebook startup or tunnel operation.\n        process = subprocess.Popen(command, cwd=self.root, stdout=writer,\n                                   stderr=subprocess.STDOUT, env=env)\n        self.processes[name] = process\n        self.readers[name] = path.open(\'r\', encoding=\'utf-8\', errors=\'replace\')\n        return process\n\n    def pump_logs(self):\n        for name, reader in self.readers.items():\n            chunk = reader.read(65536)\n            if chunk:\n                print(f\'[{name}] {chunk}\', end=\'\' if chunk.endswith(\'\\n\') else \'\\n\', flush=True)\n                if name == \'tunnel\':\n                    self.tunnel_tail = (self.tunnel_tail + chunk)[-65536:]\n                    match = re.search(r\'https://[a-zA-Z0-9-]+\\.trycloudflare\\.com\', self.tunnel_tail)\n                    if match:\n                        self.tunnel_url = match.group(0)\n\n    def _check_processes(self):\n        for name, process in self.processes.items():\n            code = process.poll()\n            if code is not None:\n                raise RuntimeError(f\'{name} exited with code {code}. See {self.root / "bridge_logs" / (name + ".log")}\')\n\n    def wait_for_ready(self):\n        deadline = time.monotonic() + self.startup_timeout\n        while time.monotonic() < deadline:\n            self.pump_logs()\n            self._check_processes()\n            try:\n                with urllib.request.urlopen(self.endpoint(path=\'/health\'), timeout=2) as response:\n                    status = json.load(response)\n                    if status.get(\'ready\') is True:\n                        # A process that wins a bind race must not be published\n                        # as this launcher\'s successfully started GPU server.\n                        if status.get(\'session_id\') != self.session_id:\n                            raise RuntimeError(\'Health endpoint belongs to a different server session.\')\n                        if self.backend != \'sd-turbo\' and status.get(\'backend\') != self.backend:\n                            raise RuntimeError(f\'{self.backend} was requested but the server reports a different backend.\')\n                        if self.diffusion_model == \'sdxl-turbo\' and status.get(\'diffusion_model\') != self.diffusion_model:\n                            raise RuntimeError(\'SDXL-Turbo was requested but the server reports a different model.\')\n                        if (self._expected_prompt_seed_model == \'sdxl-turbo\'\n                                and status.get(\'world_seed_model\') != \'sdxl-turbo\'):\n                            raise RuntimeError(\'SDXL-Turbo prompt seeds were requested but are not active; update the Waypoint server scripts.\')\n                        if self.tensorrt and not str(status.get(\'engine\', \'\')).startswith(\'TensorRT\'):\n                            raise RuntimeError(\'TensorRT was requested but is not active. Inspect bridge_logs/server.log; no endpoint was published.\')\n                        return\n            except (OSError, ValueError, urllib.error.URLError):\n                pass\n            time.sleep(0.2)\n        raise TimeoutError(\'Server startup timed out. Review bridge_logs/server.log; TensorRT compilation may need more time.\')\n\n    def wait_for_tunnel(self):\n        deadline = time.monotonic() + self.tunnel_timeout\n        while time.monotonic() < deadline:\n            self.pump_logs()\n            self._check_processes()\n            if self.tunnel_url:\n                return self.tunnel_url.replace(\'https://\', \'wss://\') + \'/ws/stream\'\n            time.sleep(0.2)\n        raise TimeoutError(\'Cloudflare did not provide a tunnel URL. Review bridge_logs/tunnel.log.\')\n\n    def start(self):\n        self.stop()\n        self.tunnel_url = None\n        self.tunnel_tail = \'\'\n        self._expected_prompt_seed_model = None\n        try:\n            with socket.create_connection((self.health_host, self.port), timeout=0.5):\n                raise RuntimeError(f\'Port {self.port} is already in use. Stop the previous server before launching this session.\')\n        except OSError:\n            pass\n        try:\n            environment = os.environ.copy()\n            for key, value in self.server_environment.items():\n                if value is None:\n                    environment.pop(key, None)\n                else:\n                    environment[key] = value\n            environment[\'BRIDGE_BACKEND\'] = self.backend\n            if self.backend in (\'krea\', \'ltx\'):\n                from krea_quant import configure_allocator\n                configure_allocator(environment)\n                # Read by Hub at import time in this child, before any downloads.\n                environment[\'HF_HUB_DISABLE_SYMLINKS\'] = \'1\'\n                environment[\'HF_HUB_DISABLE_SHARED_BLOBS\'] = \'1\'\n            environment[\'BRIDGE_DIFFUSION_MODEL\'] = self.diffusion_model\n            if self.backend == \'waypoint\' and environment.get(\'WAYPOINT_PROMPT_SEEDS\') == \'1\':\n                selected = environment.get(\'WAYPOINT_PROMPT_SEED_MODEL\', \'sd-turbo\')\n                if selected not in (\'sd-turbo\', \'sdxl-turbo\'):\n                    raise ValueError(\'Prompt seed model must be sd-turbo or sdxl-turbo\')\n                self._expected_prompt_seed_model = selected\n            environment[\'MAX_STREAM_FPS\'] = str(self.max_fps)\n            environment[\'ENABLE_TENSORRT\'] = \'1\' if self.tensorrt else \'0\'\n            environment[\'BRIDGE_HOST\'] = self.host\n            environment[\'BRIDGE_PORT\'] = str(self.port)\n            environment[\'BRIDGE_SESSION_ID\'] = self.session_id\n            environment[\'BRIDGE_INITIAL_WIDTH\'], environment[\'BRIDGE_INITIAL_HEIGHT\'] = map(str, self.output_size)\n            environment[\'BRIDGE_TRT_ALLOW_RESIZE\'] = \'1\' if self.allow_tensorrt_resize else \'0\'\n            environment[\'BRIDGE_TRT_CACHE_DIR\'] = str(self.engine_cache)\n            environment[\'BRIDGE_TRT_ENGINE_REUSE\'] = \'1\' if self.reuse_engines else \'0\'\n            if self.access_token is not None:\n                environment[\'BRIDGE_ACCESS_TOKEN\'] = self.access_token\n            if self.tensorrt:\n                if Path(self.python_executable).resolve() == Path(sys.executable).resolve():\n                    from tensorrt_backend import ensure_dependencies\n                    ensure_dependencies(self.root)\n                else:\n                    subprocess.run([self.python_executable, \'-c\',\n                                    \'import sys; sys.path.insert(0, sys.argv[1]); \'\n                                    \'from tensorrt_backend import ensure_dependencies; ensure_dependencies(sys.argv[2])\',\n                                    str(self.server_script.parent), str(self.root)], check=True, env=environment)\n            self._spawn(\'server\', [self.python_executable, \'-u\', str(self.server_script)], environment)\n            self.wait_for_ready()\n            if self.tunnel:\n                print(\'Cloudflare tunnel protocol:\', self.tunnel_protocol, flush=True)\n                self._spawn(\'tunnel\', [self.cloudflared, \'tunnel\', \'--protocol\',\n                                       self.tunnel_protocol, \'--url\', self.endpoint()])\n                return self.wait_for_tunnel()\n            return self.endpoint(self.advertise_host, \'ws\', \'/ws/stream\')\n        except BaseException:\n            self.stop()\n            raise\n\n    def wait(self):\n        while self.processes:\n            self.pump_logs()\n            self._check_processes()\n            time.sleep(0.2)\n\n    def stop(self):\n        # Only stop children owned by this instance, rather than killing any\n        # unrelated process that happens to hold port 8000.\n        for process in reversed(list(self.processes.values())):\n            if process.poll() is None:\n                try:\n                    process.terminate()\n                    process.wait(timeout=5)\n                except subprocess.TimeoutExpired:\n                    process.kill()\n                    process.wait(timeout=5)\n                except ProcessLookupError:\n                    pass\n        self.processes.clear()\n        for handle in list(self.readers.values()) + list(self.writers.values()):\n            handle.close()\n        self.readers.clear()\n        self.writers.clear()\n', 'tensorrt_backend.py': '"""Torch-TensorRT acceleration for current Colab Python/CUDA runtimes."""\nfrom pathlib import Path\nimport hashlib\nfrom importlib.metadata import version, PackageNotFoundError\nimport json\nimport logging\nimport os\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport weakref\n\n_weight_hashes = weakref.WeakKeyDictionary()\nENGINE_CACHE_SCHEMA = 1\n\n\ndef weight_fingerprint(unet):\n    signature = tuple((name, value.data_ptr(), value._version, str(value.dtype), tuple(value.shape))\n                      for name, value in list(unet.named_parameters()) + list(unet.named_buffers()))\n    previous = _weight_hashes.get(unet)\n    if previous is not None and previous[0] == signature:\n        return previous[1]\n    digest = hashlib.sha256()\n    for name, value in sorted(unet.state_dict().items()):\n        digest.update(f\'{name}:{value.dtype}:{tuple(value.shape)}\'.encode())\n        import torch\n        digest.update(value.detach().cpu().contiguous().reshape(-1).view(torch.uint8).numpy().tobytes())\n    result = digest.hexdigest()\n    _weight_hashes[unet] = (signature, result)\n    return result\n\n\ndef file_digest(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for chunk in iter(lambda: handle.read(8 * 1024**2), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef engine_metadata(unet, width, height, batch_size, torch_version, cuda_version,\n                    torchtrt_version, trt_version, gpu_name, capability):\n    try:\n        diffusers_version = version(\'diffusers\')\n    except PackageNotFoundError:\n        diffusers_version = \'unavailable\'\n    config = dict(unet.config) if hasattr(unet.config, \'items\') else vars(unet.config)\n    config = {key: value for key, value in config.items() if not key.startswith(\'_\')}\n    return dict(schema=ENGINE_CACHE_SCHEMA, weights=weight_fingerprint(unet), width=width, height=height,\n                batch=batch_size, dtype=str(unet.dtype), torch=torch_version, cuda=cuda_version,\n                torchtrt=torchtrt_version, tensorrt=trt_version, gpu=gpu_name, sm=list(capability),\n                platform=platform.system(), machine=platform.machine(), python=platform.python_version(),\n                diffusers=diffusers_version, config=hashlib.sha256(json.dumps(\n                    config, sort_keys=True, default=str).encode()).hexdigest(),\n                compiler=\'immutable-fp16-standard-attention-opt0\')\n\n\ndef alternate_inputs(inputs):\n    values = [inputs[0] * 0.5, inputs[1] * 0.5, inputs[2] * 0.5]\n    if len(inputs) == 5:\n        values.extend([inputs[3] * 0.5, inputs[4]])\n    return values\n\n\ndef load_engine_archive(directory, metadata, inputs, expected, reference=None):\n    """Load a full-weight immutable archive; never refit cached engines."""\n    import torch\n    archive, manifest = directory / \'unet.ts\', directory / \'manifest.json\'\n    if not archive.exists() or not manifest.exists():\n        return None\n    try:\n        record = json.loads(manifest.read_text(encoding=\'utf-8\'))\n        if record[\'metadata\'] != metadata:\n            return None\n        # Stage a single file on runtime disk so Drive is read only once.\n        with tempfile.TemporaryDirectory(prefix=\'bridge-trt-load-\') as name:\n            local = Path(name) / \'unet.ts\'\n            shutil.copyfile(archive, local)\n            if file_digest(local) != record[\'sha256\']:\n                raise ValueError(\'TensorRT archive checksum mismatch\')\n            module = torch.jit.load(str(local)).eval()\n        if \'tensorrt::execute_engine\' not in str(module.inlined_graph):\n            raise ValueError(\'Archive does not contain a TensorRT execution graph\')\n        torch.testing.assert_close(module(*inputs), expected, rtol=0.05, atol=0.05)\n        if reference is not None:\n            alternate = alternate_inputs(inputs)\n            torch.testing.assert_close(module(*alternate), reference(*alternate), rtol=0.05, atol=0.05)\n        torch.cuda.synchronize()\n        print(f\'>>> Loaded and validated full-weight TensorRT archive: {directory}\', flush=True)\n        return module\n    except Exception as error:\n        logging.getLogger(__name__).warning(\'TensorRT archive rejected; rebuilding: %s\', error)\n        return None\n\n\ndef save_engine_archive(module, directory, metadata, inputs, expected, reference=None):\n    import torch\n    import torch_tensorrt\n    try:\n        with tempfile.TemporaryDirectory(prefix=\'bridge-trt-save-\') as name:\n            local = Path(name) / \'unet.ts\'\n            torch_tensorrt.save(module, str(local), output_format=\'torchscript\', inputs=inputs)\n            # A successful export alone does not establish a usable cache.\n            loaded = torch.jit.load(str(local)).eval()\n            # FX can retain unused native parameters after full conversion.\n            # Freeze removes those duplicate weights from the deployment file.\n            try:\n                compact = torch.jit.freeze(loaded)\n                compact_path = Path(name) / \'compact.ts\'\n                torch.jit.save(compact, str(compact_path))\n                compact_path.replace(local)\n                loaded = compact\n            except RuntimeError as error:\n                logging.getLogger(__name__).warning(\'Archive compaction unavailable; keeping full export: %s\', error)\n            if \'tensorrt::execute_engine\' not in str(loaded.inlined_graph):\n                raise ValueError(\'Saved graph has no TensorRT execution\')\n            torch.testing.assert_close(loaded(*inputs), expected, rtol=0.05, atol=0.05)\n            if reference is not None:\n                alternate = alternate_inputs(inputs)\n                torch.testing.assert_close(loaded(*alternate), reference(*alternate), rtol=0.05, atol=0.05)\n            torch.cuda.synchronize()\n            record = dict(metadata=metadata, sha256=file_digest(local))\n            temporary = directory / (\'unet-\' + os.urandom(8).hex() + \'.tmp\')\n            try:\n                shutil.copyfile(local, temporary)\n                temporary.replace(directory / \'unet.ts\')\n            finally:\n                temporary.unlink(missing_ok=True)\n            manifest = directory / (\'manifest-\' + os.urandom(8).hex() + \'.tmp\')\n            try:\n                manifest.write_text(json.dumps(record, sort_keys=True), encoding=\'utf-8\')\n                manifest.replace(directory / \'manifest.json\')\n            finally:\n                manifest.unlink(missing_ok=True)\n        print(f\'>>> Saved validated full-weight TensorRT archive: {directory}\', flush=True)\n    except Exception as error:\n        logging.getLogger(__name__).warning(\'This engine works, but could not be cached; next launch will rebuild: %s\', error)\n\n\ndef package_requirement(torch_version, python_version=None):\n    python_version = python_version or sys.version_info[:2]\n    if tuple(python_version) < (3, 11):\n        raise RuntimeError(\'TensorRT acceleration needs Python 3.11 or newer. Use a current Colab runtime or disable TensorRT.\')\n    match = re.match(r\'^(\\d+)\\.(\\d+)\\.\', torch_version)\n    if match is None or tuple(map(int, match.groups())) < (2, 11):\n        raise RuntimeError(\'This TensorRT backend requires PyTorch 2.11 or newer. Disable TensorRT on older runtimes.\')\n    return f\'torch-tensorrt=={match[1]}.{match[2]}.*\'\n\n\ndef ensure_dependencies(root):\n    """Install matching wheels and verify imports; never silently request FP16."""\n    import torch\n    requirement = package_requirement(torch.__version__)\n    if not torch.cuda.is_available() or not torch.version.cuda:\n        raise RuntimeError(\'TensorRT requires a CUDA GPU runtime.\')\n    probe = [sys.executable, \'-c\', \'import torch_tensorrt, tensorrt; print("Torch-TensorRT", torch_tensorrt.__version__, "TensorRT", tensorrt.__version__)\']\n    result = subprocess.run(probe, cwd=root, capture_output=True, text=True)\n    if result.returncode == 0:\n        print(result.stdout.strip(), flush=True)\n        return\n    logs = Path(root) / \'bridge_logs\'\n    logs.mkdir(exist_ok=True)\n    log_path = logs / \'tensorrt-setup.log\'\n    cuda_tag = \'cu\' + torch.version.cuda.replace(\'.\', \'\')\n    command = [sys.executable, \'-m\', \'pip\', \'install\', requirement, f\'torch=={torch.__version__}\',\n               \'--extra-index-url\', f\'https://download.pytorch.org/whl/{cuda_tag}\']\n    print(f\'Installing {requirement} for PyTorch {torch.__version__} / CUDA {torch.version.cuda}.\', flush=True)\n    print(f\'Installation log: {log_path}\', flush=True)\n    with log_path.open(\'w\', encoding=\'utf-8\') as log:\n        installed = subprocess.run(command, cwd=root, stdout=log, stderr=subprocess.STDOUT, timeout=900)\n    if installed.returncode:\n        print(log_path.read_text(encoding=\'utf-8\', errors=\'replace\')[-10000:], flush=True)\n        raise RuntimeError(\'TensorRT installation failed. See bridge_logs/tensorrt-setup.log; uncheck TensorRT to use PyTorch.\')\n    result = subprocess.run(probe, cwd=root, capture_output=True, text=True)\n    if result.returncode:\n        print(result.stderr, flush=True)\n        raise RuntimeError(\'TensorRT imports failed after installation. See the startup output; acceleration was not enabled.\')\n    print(result.stdout.strip(), flush=True)\n\n\ndef cache_directory(root, width, height, batch_size, torch_version, trt_version, capability):\n    tag = re.sub(r\'[^A-Za-z0-9._-]\', \'_\', f\'torch-{torch_version}-trt-{trt_version}-sm{capability[0]}{capability[1]}\')\n    return Path(root) / tag / f\'{width}x{height}\' / f\'batch-{batch_size}\'\n\n\nclass TensorRTUNet:\n    """Adapt the compiled tensor output to StreamDiffusion\'s UNet call contract."""\n    backend = \'TensorRT (UNet)\'\n\n    def __init__(self, compiled, source, batch_size):\n        self.compiled = compiled\n        self.config = source.config\n        self.dtype = source.dtype\n        self.device = source.device\n        self.batch_size = batch_size\n\n    def __call__(self, sample, timestep, encoder_hidden_states, **kwargs):\n        import torch\n        if sample.shape[0] != self.batch_size:\n            raise ValueError(\'TensorRT UNet batch size does not match the compiled mode\')\n        inputs = [sample, timestep.to(dtype=torch.float32), encoder_hidden_states]\n        if getattr(self.config, \'addition_embed_type\', None) == \'text_time\':\n            conditioning = kwargs.get(\'added_cond_kwargs\') or {}\n            if any(key not in conditioning for key in (\'text_embeds\', \'time_ids\')):\n                raise ValueError(\'SDXL TensorRT requires live pooled text and image-size conditioning\')\n            inputs.extend([conditioning[\'text_embeds\'], conditioning[\'time_ids\']])\n        result = self.compiled(*inputs)\n        return (result,)\n\n    def to(self, *args, **kwargs):\n        return self\n\n    def eval(self):\n        return self\n\n\ndef compile_unet(unet, width, height, batch_size, cache_root=None):\n    import torch\n    import torch_tensorrt\n    import tensorrt\n    logging.getLogger(\'torch_tensorrt.dynamo.conversion\').setLevel(logging.WARNING)\n    logging.getLogger(\'torch_tensorrt.dynamo.conversion._conversion\').setLevel(logging.INFO)\n    package_requirement(torch.__version__)\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'TensorRT compilation requires a CUDA GPU.\')\n\n    class UNetGraph(torch.nn.Module):\n        def __init__(self, model):\n            super().__init__()\n            self.model = model\n\n        def forward(self, sample, timestep, encoder_hidden_states):\n            return self.model(sample, timestep, encoder_hidden_states=encoder_hidden_states, return_dict=False)[0]\n\n    class XLUNetGraph(UNetGraph):\n        def forward(self, sample, timestep, encoder_hidden_states, text_embeds, time_ids):\n            return self.model(sample, timestep, encoder_hidden_states=encoder_hidden_states,\n                              added_cond_kwargs={\'text_embeds\': text_embeds, \'time_ids\': time_ids},\n                              return_dict=False)[0]\n\n    is_xl = getattr(unet.config, \'addition_embed_type\', None) == \'text_time\'\n    graph = (XLUNetGraph(unet) if is_xl else UNetGraph(unet)).eval()\n    reuse_engines = os.environ.get(\'BRIDGE_TRT_ENGINE_REUSE\', \'1\') == \'1\'\n    cache_root = cache_root or os.environ.get(\'BRIDGE_TRT_CACHE_DIR\', \'engines/torch-tensorrt\')\n    inputs = [\n        torch.randn((batch_size, unet.config.in_channels, height // 8, width // 8), device=unet.device, dtype=unet.dtype),\n        torch.full((batch_size,), 999.0, device=unet.device, dtype=torch.float32),\n        torch.randn((batch_size, 77, unet.config.cross_attention_dim), device=unet.device, dtype=unet.dtype),\n    ]\n    if is_xl:\n        pooled_dim = unet.config.projection_class_embeddings_input_dim - 6 * unet.config.addition_time_embed_dim\n        if pooled_dim <= 0:\n            raise ValueError(\'Invalid SDXL pooled text dimension\')\n        inputs.extend([\n            torch.randn((batch_size, pooled_dim), device=unet.device, dtype=unet.dtype),\n            torch.tensor([[height, width, 0, 0, height, width]], device=unet.device,\n                         dtype=unet.dtype).repeat(batch_size, 1),\n        ])\n    cache = cache_directory(cache_root, width, height, batch_size, torch.__version__,\n                            f\'{torch_tensorrt.__version__}-sdk-{tensorrt.__version__}\',\n                            torch.cuda.get_device_capability())\n    metadata = engine_metadata(unet, width, height, batch_size, torch.__version__, torch.version.cuda,\n                               torch_tensorrt.__version__, tensorrt.__version__,\n                               torch.cuda.get_device_name(), torch.cuda.get_device_capability())\n    key = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()[:24]\n    cache = cache / key\n    cache.mkdir(parents=True, exist_ok=True)\n    print(f\'>>> Checking TensorRT profile: {width} x {height}, batch {batch_size}.\', flush=True)\n    with torch.no_grad():\n        expected = graph(*inputs).detach()\n        if reuse_engines:\n            loaded = load_engine_archive(cache, metadata, inputs, expected, graph)\n            if loaded is not None:\n                return TensorRTUNet(loaded, unet, batch_size)\n        print(f\'>>> Compiling TensorRT UNet: {width} x {height}, batch {batch_size}. First use can take several minutes.\', flush=True)\n        # Export standard attention operations: FlashAttention cross-attention\n        # lacks converter coverage, and its self-attention converter requires\n        # strong typing that triggers a TensorRT 10.15 builder error here.\n        # Compare the compiled result against the original attention processor.\n        from diffusers.models.attention_processor import AttnProcessor\n        unet.set_attn_processor({\n            name: AttnProcessor() for name in unet.attn_processors\n        })\n        compiled = torch_tensorrt.compile(\n            # This FP16 builder path avoids the TensorRT 10.15 strong-typing\n            # compiler failure while retaining full TensorRT graph coverage.\n            graph, ir=\'dynamo\', inputs=inputs, use_explicit_typing=False,\n            enabled_precisions={torch.float16}, optimization_level=0,\n            # Decompose Diffusers\' baddbmm into supported primitive operations.\n            enable_experimental_decompositions=True,\n            use_python_runtime=False,\n            require_full_compilation=True, workspace_size=2 * 1024**3,\n            # TensorRT 10.15 refit returned NaNs after a real cache hit on\n            # SD-Turbo. Keep immutable weights in full deployment archives;\n            # the compiler\'s stripped/refit cache must remain disabled.\n            immutable_weights=True,\n            cache_built_engines=False, reuse_cached_engines=False,\n            timing_cache_path=str(cache / \'timing-cache.bin\'),\n        )\n        engine_count = sum(type(module).__name__ in (\'TorchTensorRTModule\', \'PythonTorchTensorRTModule\')\n                           for module in compiled.modules())\n        if engine_count == 0:\n            raise RuntimeError(\'Compilation produced no TensorRT engines; acceleration was not enabled.\')\n        actual = compiled(*inputs)\n        torch.testing.assert_close(actual, expected, rtol=0.05, atol=0.05)\n        torch.cuda.synchronize()\n        # Check another timestep/input before exporting or reporting readiness.\n        # Prompt conditioning stays a graph input, never frozen in a cache.\n        check_inputs = alternate_inputs(inputs)\n        torch.testing.assert_close(compiled(*check_inputs), graph(*check_inputs), rtol=0.05, atol=0.05)\n        if reuse_engines:\n            save_engine_archive(compiled, cache, metadata, inputs, expected, graph)\n    print(f\'>>> TensorRT UNet batch {batch_size} compiled and output validation passed ({engine_count} engine(s)).\', flush=True)\n    return TensorRTUNet(compiled, unet, batch_size)\n', 'colab_setup.py': '"""Colab setup source embedded into the generated notebook."""\nimport os\nfrom pathlib import Path\nimport platform\nimport subprocess\nimport sys\nimport tempfile\nimport urllib.request\n\nUPSTREAM_REVISION = \'b623251dc055e1fd858d53509aa43e09dfc5cdc0\'\nCORE_DEPENDENCIES = (\n    \'diffusers==0.24.0\', \'transformers==4.39.3\',\n    \'huggingface-hub==0.25.2\', \'accelerate==0.30.1\',\n    \'fastapi\', \'uvicorn\', \'websockets\', \'python-multipart\',\n    \'opencv-python-headless\', \'pillow\',\n)\nDEFAULT_DRIVE_MODEL_FOLDER = \'StreamDiffusion/models\'\nMODEL_CACHE_VARIABLES = (\'HF_HUB_CACHE\', \'HUGGINGFACE_HUB_CACHE\', \'TRANSFORMERS_CACHE\')\n\n\ndef configure_model_cache(use_google_drive=False, drive_folder=DEFAULT_DRIVE_MODEL_FOLDER,\n                          mount_point=\'/content/drive\', local_cache=None):\n    """Select model storage before launching the server\'s fresh Python process."""\n    if use_google_drive:\n        folder = Path(drive_folder)\n        if not str(drive_folder).strip() or not folder.parts or folder.is_absolute() or \'..\' in folder.parts:\n            raise ValueError(\'Drive model folder must be a relative folder inside MyDrive, without ..\')\n        from google.colab import drive\n        drive.mount(str(mount_point))\n        my_drive = (Path(mount_point) / \'MyDrive\').resolve()\n        if not my_drive.is_dir():\n            raise RuntimeError(\'Google Drive is not mounted. Authorize Drive access and rerun setup.\')\n        cache = (my_drive / folder).resolve()\n        if not cache.is_relative_to(my_drive) or cache == my_drive:\n            raise ValueError(\'Choose a dedicated model cache folder inside MyDrive.\')\n    else:\n        cache = Path(local_cache or Path.home() / \'.cache\' / \'huggingface\' / \'hub\').expanduser().resolve()\n    cache.mkdir(parents=True, exist_ok=True)\n    # Fail visibly on a disconnected/read-only/full Drive instead of claiming\n    # persistence while downloading models into the temporary VM filesystem.\n    with tempfile.NamedTemporaryFile(prefix=\'.bridge-cache-check-\', dir=cache) as probe:\n        probe.write(b\'model-cache-write-check\')\n        probe.flush()\n    for variable in MODEL_CACHE_VARIABLES:\n        os.environ[variable] = str(cache)\n    # Leave HF_HOME and HF_TOKEN_PATH alone: model persistence does not require\n    # moving authentication tokens into the user\'s Drive model folder.\n    location = \'Google Drive (persistent)\' if use_google_drive else \'runtime disk (temporary)\'\n    print(f\'Model cache: {cache} [{location}]\', flush=True)\n    if use_google_drive:\n        print(\'Existing cached weights are reused. Missing or updated model files download here on first use.\', flush=True)\n    return cache\n\n\ndef setup_environment(root=\'/content/StreamDiffusion\', use_google_drive=False,\n                      drive_model_folder=DEFAULT_DRIVE_MODEL_FOLDER):\n    if platform.system() != \'Linux\':\n        raise RuntimeError(\'This setup targets Google Colab Linux GPU runtimes.\')\n    configure_model_cache(use_google_drive, drive_model_folder)\n    root = Path(root).resolve()\n    if not root.exists():\n        subprocess.run([\'git\', \'clone\', \'https://github.com/cumulo-autumn/StreamDiffusion.git\', str(root)], check=True)\n    subprocess.run([\'git\', \'-C\', str(root), \'fetch\', \'--depth\', \'1\', \'origin\', UPSTREAM_REVISION], check=True)\n    subprocess.run([\'git\', \'-C\', str(root), \'checkout\', \'--detach\', UPSTREAM_REVISION], check=True)\n    os.chdir(root)\n    # Resolve the editable package and compatibility pins together so a later\n    # install cannot silently undo earlier transformer/hub constraints.\n    subprocess.run([sys.executable, \'-m\', \'pip\', \'uninstall\', \'-y\', \'peft\'], check=True)\n    subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'-e\', \'.\', *CORE_DEPENDENCIES], check=True)\n    target = root / \'cloudflared\'\n    if not target.exists():\n        temporary = root / \'cloudflared.download\'\n        try:\n            urllib.request.urlretrieve(\'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\', temporary)\n            temporary.replace(target)\n        finally:\n            temporary.unlink(missing_ok=True)\n    target.chmod(0o755)\n    print(\'Setup complete. StreamDiffusion revision:\', UPSTREAM_REVISION)\n    print(\'Run the next cell to write the bridge server and helpers.\')\n    return root\n\n\nif __name__ == \'__main__\':\n    # Colab\'s generated setup cell defines these form values above this source.\n    # Importing this module for workstation setup never mounts Drive.\n    setup_environment(use_google_drive=globals().get(\'USE_GOOGLE_DRIVE\', False),\n                      drive_model_folder=globals().get(\'DRIVE_MODEL_FOLDER\', DEFAULT_DRIVE_MODEL_FOLDER))\n', 'waypoint_backend.py': '"""Waypoint public-API adapter; the separately installed world_engine owns inference."""\nfrom collections import deque\nimport inspect\nimport logging\nimport math\nimport threading\nimport time\n\nimport cv2\nimport numpy as np\n\nfrom bridge_protocol import integer\n\nWORLD_SIZE = (1280, 720)\nRUNTIME_REVISION = \'68b316390d576dd9f671f552803526bbf8eb2ed5\'\nMODEL_ID = \'Overworld/Waypoint-1.5-1B\'\nMODEL_REVISION = \'391f92827075edcf4a8b3c8a2ddae010698f8636\'\nVAE_ID = \'Overworld-Models/taehv1_5\'\nVAE_REVISION = \'a0253886b13b9c4c3bd224bd479be03f5988a3df\'\nlogger = logging.getLogger(__name__)\n\n\ndef engine_class_for_profile(base, torch_module, profile):\n    """Override only compiler policy; keep the pinned runtime\'s model operations."""\n    if profile == \'max-autotune\':\n        return base\n    if profile != \'fast\':\n        raise ValueError(\'Compile profile must be fast or max-autotune\')\n    class BridgeWorldEngine(base):\n        pass\n    for name in (\'_cache_pass\', \'_denoise_pass\'):\n        # torch.compile uses functools.wraps; unwrap at the class level to\n        # preserve the original self argument, then compile the same function.\n        original = inspect.unwrap(getattr(base, name))\n        if not inspect.isfunction(original):\n            raise RuntimeError(f\'Unsupported world_engine compiler method: {name}\')\n        setattr(BridgeWorldEngine, name, torch_module.compile(\n            original, fullgraph=True, dynamic=False,\n            options={\'max_autotune\': False, \'coordinate_descent_tuning\': False,\n                     \'triton.cudagraphs\': False}))\n    return BridgeWorldEngine\n\n\ndef validate_world_controls(data, current=WORLD_SIZE):\n    """Accept only this checkpoint\'s native dimensions and supported controls."""\n    if not isinstance(data, dict):\n        raise ValueError(\'Control must be an object\')\n    if (integer(data.get(\'width\', current[0]), \'width\'),\n            integer(data.get(\'height\', current[1]), \'height\')) != WORLD_SIZE:\n        raise ValueError(\'Waypoint uses native 1280 x 720 output\')\n    if data.get(\'mode\', \'world\') != \'world\':\n        raise ValueError(\'Select World Model for this server\')\n    clean = {}\n    if \'world_buttons\' in data:\n        buttons = data[\'world_buttons\']\n        if not isinstance(buttons, list) or len(buttons) > 32:\n            raise ValueError(\'Use a list of at most 32 Owl-Control button codes\')\n        buttons = [integer(value, \'button\') for value in buttons]\n        if any(not 0 <= value < 256 for value in buttons):\n            raise ValueError(\'Button codes must be between 0 and 255\')\n        clean[\'world_buttons\'] = sorted(set(buttons))\n    for name in (\'world_mouse_x\', \'world_mouse_y\'):\n        if name in data:\n            value = data[name]\n            if (isinstance(value, bool) or not isinstance(value, (float, int))\n                    or not math.isfinite(value) or not -1 <= value <= 1):\n                raise ValueError(\'Camera motion must be between -1 and 1\')\n            clean[name] = float(value)\n    for name in (\'world_reset\', \'world_paused\'):\n        if name in data:\n            if not isinstance(data[name], bool):\n                raise ValueError(f\'{name} must be a boolean\')\n            clean[name] = data[name]\n    if \'seed\' in data:\n        seed = integer(data[\'seed\'], \'seed\')\n        if not 0 <= seed <= 4294967295:\n            raise ValueError(\'seed must be between 0 and 4294967295\')\n        clean[\'seed\'] = seed\n    if \'world_seed_source\' in data:\n        if data[\'world_seed_source\'] not in (\'input\', \'sdturbo\'):\n            raise ValueError(\'World seed source must be input or sdturbo\')\n        clean[\'world_seed_source\'] = data[\'world_seed_source\']\n    for name in (\'world_auto_seed\', \'world_reseed\'):\n        if name in data:\n            if not isinstance(data[name], bool):\n                raise ValueError(f\'{name} must be a boolean\')\n            clean[name] = data[name]\n    for name, low, high in ((\'world_seed_interval\', 1, 3600), (\'blend\', 0, 1)):\n        if name in data:\n            value = data[name]\n            if (isinstance(value, bool) or not isinstance(value, (float, int))\n                    or not math.isfinite(value) or not low <= value <= high):\n                raise ValueError(f\'{name} must be between {low} and {high}\')\n            clean[name] = float(value)\n    for name in (\'prompt_a\', \'prompt_b\'):\n        if name in data:\n            value = data[name]\n            if not isinstance(value, str) or len(value) > 4096 or (name == \'prompt_a\' and not value.strip()):\n                raise ValueError(f\'{name} must be valid prompt text\')\n            clean[name] = value.strip()\n    return clean\n\n\nclass WaypointBackend:\n    """Single-worker state: at most one four-frame batch, no unbounded prefetch."""\n    def __init__(self, engine, ctrl_type, torch_module, seed=42, seed_generator=None, prompt_seed_model=\'sd-turbo\'):\n        from waypoint_setup import PROMPT_SEED_MODELS\n        if prompt_seed_model not in PROMPT_SEED_MODELS:\n            raise ValueError(\'Prompt seed model must be sd-turbo or sdxl-turbo\')\n        self.engine, self.ctrl_type, self.torch = engine, ctrl_type, torch_module\n        self.frames = deque()\n        self.seed_image = None\n        self.seed = seed\n        self.buttons, self.mouse = [], [0.0, 0.0]\n        self.paused = False\n        self.batch_ms = 0.0\n        self._progress_lock = threading.Lock()\n        self._phase, self._phase_started = \'awaiting_seed\', time.monotonic()\n        self.batches = 0\n        self.compile_profile = \'fast\'\n        self.seed_generator = seed_generator\n        self.prompt_seed_model = prompt_seed_model\n        self.prompt_seed_label = \'SDXL-Turbo\' if prompt_seed_model == \'sdxl-turbo\' else \'SD-Turbo\'\n        self.seed_source, self.auto_seed, self.seed_interval = \'input\', True, 5.0\n        self.prompt_a, self.prompt_b, self.blend = \'cinematic landscape, detailed\', \'\', 0.0\n        self.seed_count, self.seed_ms, self.seed_error = 0, 0.0, None\n        self._reseed_pending, self._seed_deadline = False, None\n\n    def _set_phase(self, phase):\n        with self._progress_lock:\n            changed = phase != self._phase\n            if changed:\n                self._phase, self._phase_started = phase, time.monotonic()\n        if changed:\n            logger.info(\'Waypoint phase: %s\', phase)\n\n    def progress(self):\n        # Called by the event loop while the single GPU worker is compiling.\n        with self._progress_lock:\n            phase, elapsed = self._phase, time.monotonic() - self._phase_started\n            batches = self.batches\n        labels = {\n            \'awaiting_seed\': \'Connected; World > Seed World from Input\',\n            \'seeding\': \'Preparing world seed; compiling kernels if needed\',\n            \'seed_generation\': f\'{self.prompt_seed_label} generating a new world seed\',\n            \'ready\': \'World seed prepared; waiting to generate\',\n            \'first_batch\': \'Generating first world batch; compiling kernels if needed\',\n            \'streaming\': \'World frames ready\',\n            \'paused\': \'World paused\',\n            \'error\': \'World processing failed; see server log, then reset or reseed\',\n        }\n        status = labels[phase]\n        if self.seed_error:\n            status = self.seed_error\n        if phase in (\'seeding\', \'first_batch\', \'seed_generation\'):\n            status += f\' ({elapsed:.0f}s)\'\n        return {\'world_phase\': phase, \'world_status\': status,\n                \'world_phase_ms\': round(elapsed * 1000), \'world_batches\': batches,\n                \'world_compile_profile\': self.compile_profile,\n                \'world_seed_required\': self.seed_image is None,\n                \'world_prompt_seeding\': self.seed_generator is not None,\n                \'world_seed_model\': self.prompt_seed_model if self.seed_generator is not None else None,\n                \'world_seed_model_label\': self.prompt_seed_label if self.seed_generator is not None else None,\n                \'world_seed_source\': self.seed_source, \'world_auto_seed\': self.auto_seed,\n                \'world_seed_interval\': self.seed_interval, \'world_seed_count\': self.seed_count,\n                \'world_seed_ms\': round(self.seed_ms, 1), \'world_seed_error\': self.seed_error}\n\n    @classmethod\n    def load(cls, quant=\'intw8a8\', compile_profile=\'fast\', prompt_seeds=False, prompt_seed_model=\'sd-turbo\'):\n        from waypoint_setup import PROMPT_SEED_MODELS\n        if prompt_seed_model not in PROMPT_SEED_MODELS:\n            raise ValueError(\'Prompt seed model must be sd-turbo or sdxl-turbo\')\n        import torch\n        from huggingface_hub import snapshot_download\n        from world_engine import WorldEngine, CtrlInput\n        if quant not in (\'intw8a8\', \'bf16\'):\n            raise ValueError(\'Use intw8a8 or bf16 here; FP8/FP4 are not selected\')\n        engine_type = engine_class_for_profile(WorldEngine, torch, compile_profile)\n        if not torch.cuda.is_available() or torch.cuda.get_device_capability()[0] < 8:\n            raise RuntimeError(\'Waypoint requires an NVIDIA Ampere-or-newer CUDA GPU\')\n        model = snapshot_download(MODEL_ID, revision=MODEL_REVISION,\n                                  allow_patterns=[\'config.yaml\', \'model.safetensors\'])\n        vae = snapshot_download(VAE_ID, revision=VAE_REVISION,\n                                allow_patterns=[\'taehv1_5.pth\'])\n        with torch.no_grad():\n            engine = engine_type(model, quant=None if quant == \'bf16\' else quant,\n                                 model_config_overrides={\'ae_uri\': vae},\n                                 device=\'cuda\', dtype=torch.bfloat16)\n        if quant == \'intw8a8\' and compile_profile == \'fast\':\n            # On a cold cache the pinned runtime\'s quantizer selects GemLite\n            # "max" for warmup and leaves it enabled. TorchInductor options do\n            # not control this custom op\'s independent Triton autotuner.\n            # Apply this AFTER construction so warmup cannot override it.\n            import gemlite\n            gemlite.set_autotune(False)\n            logger.info(\'Waypoint fast profile: GemLite default tuning selected after INT8 warmup\')\n        seed_generator = None\n        if prompt_seeds:\n            from waypoint_seed import PromptSeedGenerator\n            seed_generator = PromptSeedGenerator.load(prompt_seed_model)\n        backend = cls(engine, CtrlInput, torch, seed_generator=seed_generator, prompt_seed_model=prompt_seed_model)\n        backend.compile_profile = compile_profile\n        logger.info(\'Waypoint compile profile: %s (native 1280 x 720, %s)\', compile_profile, quant)\n        return backend\n\n    @property\n    def can_generate(self):\n        return not self.paused and (self.seed_image is not None or (\n            self.seed_source == \'sdturbo\' and self.seed_generator is not None and self._reseed_pending))\n\n    def release_controls(self):\n        self.buttons, self.mouse = [], [0.0, 0.0]\n        self.frames.clear()\n\n    def configure(self, data):\n        clean = validate_world_controls(data)\n        source = clean.get(\'world_seed_source\', self.seed_source)\n        if source == \'sdturbo\' and self.seed_generator is None:\n            raise ValueError(\'Enable prompt seeds in Waypoint server setup first\')\n        source_changed = source != self.seed_source\n        interval_changed = clean.get(\'world_seed_interval\', self.seed_interval) != self.seed_interval\n        changed = any(clean.get(key, old) != old for key, old in (\n            (\'world_seed_source\', self.seed_source), (\'prompt_a\', self.prompt_a),\n            (\'prompt_b\', self.prompt_b), (\'blend\', self.blend), (\'seed\', self.seed)))\n        self.seed_source = source\n        for key, attr in ((\'prompt_a\', \'prompt_a\'), (\'prompt_b\', \'prompt_b\'), (\'blend\', \'blend\'),\n                          (\'world_auto_seed\', \'auto_seed\'), (\'world_seed_interval\', \'seed_interval\')):\n            if key in clean:\n                setattr(self, attr, clean[key])\n        if source == \'sdturbo\' and (source_changed or clean.get(\'world_reseed\') or (changed and self.auto_seed)):\n            self._reseed_pending, self.seed_error = True, None\n        elif source == \'input\':\n            self._reseed_pending, self.seed_error = False, None\n        if interval_changed and self._seed_deadline is not None:\n            self._seed_deadline = time.monotonic() + self.seed_interval\n        if \'world_buttons\' in clean:\n            self.buttons = clean[\'world_buttons\']\n        for index, name in enumerate((\'world_mouse_x\', \'world_mouse_y\')):\n            if name in clean:\n                self.mouse[index] = clean[name]\n        if \'world_paused\' in clean:\n            self.paused = clean[\'world_paused\']\n        if \'seed\' in clean:\n            self.seed = clean[\'seed\']\n        if clean.get(\'world_reset\'):\n            self.reset()\n        elif \'world_paused\' in clean:\n            self._set_phase(\'paused\' if self.paused else \'awaiting_seed\' if self.seed_image is None\n                            else \'streaming\' if self.batches else \'ready\')\n\n    def reset(self):\n        self.frames.clear()\n        self._seed_deadline = None\n        with self._progress_lock:\n            self.batches = 0\n        self._set_phase(\'seeding\' if self.seed_image is not None else \'awaiting_seed\')\n        self.torch.manual_seed(self.seed)\n        try:\n            self.engine.reset()\n            if self.seed_image is not None:\n                self.engine.append_frame(self.seed_image)\n            self._set_phase(\'paused\' if self.paused else \'ready\' if self.seed_image is not None else \'awaiting_seed\')\n        except Exception:\n            # Partial cache/encoder state must not be used for further inference.\n            self.seed_image = None\n            self._set_phase(\'error\')\n            raise\n\n    def set_seed_image(self, jpeg):\n        bgr = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)\n        if bgr is None:\n            raise ValueError(\'Seed must be a JPEG image\')\n        self.seed_source, self._reseed_pending, self.seed_error = \'input\', False, None\n        self.set_seed_rgb(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))\n\n    def set_seed_rgb(self, rgb):\n        if rgb.ndim != 3 or rgb.shape[2] != 3 or rgb.dtype != np.uint8:\n            raise ValueError(\'Seed must contain RGB uint8 pixels\')\n        # Fit without stretching: center crop to native 16:9, then resize.\n        height, width = rgb.shape[:2]\n        crop_w, crop_h = min(width, round(height * 16 / 9)), min(height, round(width * 9 / 16))\n        x, y = (width - crop_w) // 2, (height - crop_h) // 2\n        rgb = cv2.resize(rgb[y:y+crop_h, x:x+crop_w], WORLD_SIZE)\n        image = self.torch.from_numpy(np.repeat(rgb[None], 4, axis=0))\n        self.seed_image = image\n        self.reset()\n\n    def next_frame(self):\n        if not self.can_generate:\n            return None\n        if not self.frames:\n            due = self.auto_seed and self._seed_deadline is not None and time.monotonic() >= self._seed_deadline\n            if self.seed_source == \'sdturbo\' and (self._reseed_pending or due):\n                self._set_phase(\'seed_generation\')\n                self._reseed_pending = False\n                started = time.perf_counter()\n                try:\n                    rgb = self.seed_generator.generate(\n                        self.prompt_a, self.prompt_b, self.blend, (self.seed + self.seed_count) % 4294967296)\n                    self.set_seed_rgb(rgb)\n                    self.seed_count += 1\n                    self.seed_ms = (time.perf_counter() - started) * 1000\n                    self.seed_error = None\n                except Exception:\n                    # Pause automatic retries. An SD failure can keep an existing\n                    # world playing; a failed world reset invalidates its seed.\n                    logger.exception(\'Prompt-to-world reseed failed\')\n                    self.auto_seed, self._seed_deadline = False, None\n                    self.seed_error = \'Prompt seed failed; auto reseed stopped. Submit prompt to retry.\'\n                    self._set_phase(\'error\' if self.seed_image is None else \'streaming\')\n                    if self.seed_image is None:\n                        return None\n            started = time.perf_counter()\n            if not self.batches:\n                self._set_phase(\'first_batch\')\n            # Values are explicitly held camera velocity, sampled once per four-frame step.\n            ctrl = self.ctrl_type(button=set(self.buttons), mouse=tuple(self.mouse))\n            try:\n                batch = self.engine.gen_frame(ctrl=ctrl).detach().cpu().numpy()\n                if batch.shape != (4, 720, 1280, 3) or batch.dtype != np.uint8:\n                    raise RuntimeError(\'Waypoint returned an unexpected frame batch\')\n            except Exception:\n                self.frames.clear()\n                self.seed_image = None\n                self._set_phase(\'error\')\n                raise\n            self.batch_ms = (time.perf_counter() - started) * 1000\n            with self._progress_lock:\n                self.batches += 1\n            self.frames.extend(batch)\n            if self.seed_source == \'sdturbo\' and self._seed_deadline is None:\n                # The initial compiler pass may take minutes. Count the interval\n                # from usable output so it cannot trigger an immediate reset.\n                self._seed_deadline = time.monotonic() + self.seed_interval\n            self._set_phase(\'streaming\')\n        return cv2.cvtColor(self.frames.popleft(), cv2.COLOR_RGB2BGR), self.batch_ms\n', 'waypoint_setup.py': '"""Install the optional Waypoint runtime in its own environment, never over SD-Turbo."""\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport subprocess\nimport sys\nimport urllib.request\nimport venv\n\nRUNTIME_REVISION = \'68b316390d576dd9f671f552803526bbf8eb2ed5\'\nBOOTSTRAP_PIP = \'pip==26.3\'\nDEPENDENCIES = (\n    f\'world_engine @ git+https://github.com/Overworldai/world_engine.git@{RUNTIME_REVISION}\',\n    \'fastapi==0.162.0\', \'uvicorn==0.49.0\', \'websockets==16.0\',\n    \'opencv-python-headless==4.13.0.92\', \'pillow==13.0.0\',\n    \'transformers==5.10.1\', \'huggingface-hub==1.7.2\',\n)\n# Compatible with the existing Hub 1.7 / Transformers 5 pins. Newer Diffusers\n# require a newer Hub; do not silently upgrade the working world-model stack.\nSEED_DEPENDENCIES = (\'diffusers==0.39.0\',)\nPROMPT_SEED_MODELS = (\'sd-turbo\', \'sdxl-turbo\')\n\n\ndef _capture(command, stage):\n    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                            text=True, encoding=\'utf-8\', errors=\'replace\')\n    if result.returncode:\n        raise RuntimeError(f\'{stage} failed (exit {result.returncode}):\\n{result.stdout[-6000:]}\')\n    return result.stdout.strip()\n\n\ndef ensure_environment(environment, label=\'Waypoint\'):\n    """Create or repair pip without relying on the host\'s ensurepip package."""\n    environment = Path(environment).resolve()\n    python = environment / (\'Scripts/python.exe\' if os.name == \'nt\' else \'bin/python\')\n    created = not python.exists() or not (environment / \'pyvenv.cfg\').is_file()\n    if created:\n        venv.EnvBuilder(with_pip=False).create(environment)\n    # Check the target before allowing host pip to install into it. A venv\'s\n    # interpreter may be a symlink, so inspect sys.prefix rather than resolving it.\n    state = json.loads(_capture([str(python), \'-I\', \'-c\',\n        \'import json, sys; print(json.dumps({"prefix": sys.prefix, \'\n        \'"base": sys.base_prefix, "version": list(sys.version_info[:2])}))\'],\n        \'Virtual environment check\'))\n    if (Path(state[\'prefix\']).resolve() != environment\n            or state[\'prefix\'] == state[\'base\']\n            or state[\'version\'] != list(sys.version_info[:2])):\n        raise RuntimeError(f\'{label} Python is not an isolated environment for this Python version; \'\n                           \'use a fresh runtime or a different runtime folder\')\n    probe = subprocess.run([str(python), \'-I\', \'-m\', \'pip\', \'--version\'],\n                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n    if probe.returncode == 0:\n        return python, created\n    if not created:\n        # An ensurepip failure leaves python/pyvenv.cfg but may omit activation\n        # scripts. Complete these files in place; preserve installed packages.\n        venv.EnvBuilder(with_pip=False).create(environment)\n    print(f\'Bootstrapping pip in the isolated {label} environment (without ensurepip)...\', flush=True)\n    output = _capture([sys.executable, \'-m\', \'pip\', \'--python\', str(python),\n                       \'install\', \'--no-cache-dir\', \'--force-reinstall\', BOOTSTRAP_PIP], \'Isolated pip bootstrap\')\n    if output:\n        print(output, flush=True)\n    _capture([str(python), \'-I\', \'-m\', \'pip\', \'--version\'], \'Isolated pip verification\')\n    return python, True\n\n\ndef setup(root, prompt_seeds=False, install_tunnel=True):\n    if platform.system() not in (\'Linux\', \'Windows\') or platform.machine().lower() not in (\'x86_64\', \'amd64\'):\n        raise RuntimeError(\'This installer supports x86-64 Linux and Windows CUDA hosts\')\n    root = Path(root).resolve()\n    environment = root / \'venv\'\n    marker = root / \'installed.json\'\n    dependencies = (*DEPENDENCIES, *(SEED_DEPENDENCIES if prompt_seeds else ()))\n    signature = {\'python\': list(sys.version_info[:2]), \'dependencies\': list(dependencies)}\n    root.mkdir(parents=True, exist_ok=True)\n    python, repaired = ensure_environment(environment)\n    reusable = not repaired and marker.exists() and json.loads(marker.read_text()) == signature\n    if not reusable:\n        # Windows PyPI Torch may be CPU-only; select the CUDA build explicitly.\n        subprocess.run([str(python), \'-m\', \'pip\', \'install\', \'--no-cache-dir\',\n                        \'torch==2.11.0\', \'--index-url\', \'https://download.pytorch.org/whl/cu130\'], check=True)\n        subprocess.run([str(python), \'-m\', \'pip\', \'install\', \'--no-cache-dir\', *dependencies], check=True)\n    subprocess.run([str(python), \'-m\', \'pip\', \'check\'], check=True)\n    # Fail before model downloads when the isolated runtime cannot use CUDA.\n    subprocess.run([str(python), \'-c\',\n                    \'import torch; from world_engine import WorldEngine; \'\n                    \'assert torch.cuda.is_available(), "CUDA unavailable in Waypoint environment"; \'\n                    \'assert torch.cuda.get_device_capability()[0] >= 8, "Ampere+ GPU required"; \'\n                    \'print(torch.__version__, torch.cuda.get_device_name(), flush=True)\'], check=True)\n    marker.write_text(json.dumps(signature, indent=2) + \'\\n\', encoding=\'utf-8\')\n    if not install_tunnel:\n        return python, None  # workstation LAN mode needs no tunnel binary\n    binary = root / (\'cloudflared.exe\' if os.name == \'nt\' else \'cloudflared\')\n    if not binary.exists():\n        asset = \'cloudflared-windows-amd64.exe\' if os.name == \'nt\' else \'cloudflared-linux-amd64\'\n        partial = binary.with_suffix(\'.download\')\n        try:\n            urllib.request.urlretrieve(f\'https://github.com/cloudflare/cloudflared/releases/latest/download/{asset}\', partial)\n            partial.replace(binary)\n        finally:\n            partial.unlink(missing_ok=True)\n        binary.chmod(0o755)\n    return python, binary\n', 'waypoint_seed.py': '"""Optional Turbo seed images, using the isolated Waypoint Diffusers stack."""\nfrom collections import OrderedDict\n\nimport numpy as np\n\nMODEL_ID = \'stabilityai/sd-turbo\'\nMODEL_REVISION = \'b261bac6fd2cf515557d5d0707481eafa0485ec2\'\nXL_MODEL_ID = \'stabilityai/sdxl-turbo\'\nXL_MODEL_REVISION = \'71153311d3dbb46851df1931d3ca6e939de83304\'\nSEED_MODELS = {\'sd-turbo\': (MODEL_ID, MODEL_REVISION, \'SD-Turbo\'),\n               \'sdxl-turbo\': (XL_MODEL_ID, XL_MODEL_REVISION, \'SDXL-Turbo\')}\nSEED_SIZE = (768, 432)\n\n\nclass PromptSeedGenerator:\n    def __init__(self, pipe, torch_module, model=\'sd-turbo\', device=\'cuda\'):\n        if model not in SEED_MODELS:\n            raise ValueError(\'Prompt seed model must be sd-turbo or sdxl-turbo\')\n        self.pipe, self.torch = pipe, torch_module\n        self.model, self.label, self.device = model, SEED_MODELS[model][2], device\n        self.embeddings = OrderedDict()\n\n    @classmethod\n    def load(cls, model=\'sd-turbo\'):\n        if model not in SEED_MODELS:\n            raise ValueError(\'Prompt seed model must be sd-turbo or sdxl-turbo\')\n        import torch\n        if model == \'sdxl-turbo\':\n            from diffusers import StableDiffusionXLPipeline as pipeline_type\n            options = {\'add_watermarker\': False}\n        else:\n            from diffusers import StableDiffusionPipeline as pipeline_type\n            options = {\'safety_checker\': None}\n        model_id, revision, _ = SEED_MODELS[model]\n        pipe = pipeline_type.from_pretrained(\n            model_id, revision=revision, variant=\'fp16\',\n            torch_dtype=torch.float16, use_safetensors=True, **options)\n        pipe.to(\'cuda\')\n        pipe.set_progress_bar_config(disable=True)\n        return cls(pipe, torch, model)\n\n    def _embedding(self, prompt):\n        value = self.embeddings.pop(prompt, None)\n        if value is None:\n            encoded = self.pipe.encode_prompt(\n                prompt, device=self.device, num_images_per_prompt=1,\n                do_classifier_free_guidance=False)\n            value = (encoded[0], encoded[2]) if self.model == \'sdxl-turbo\' else encoded[0]\n        self.embeddings[prompt] = value\n        while len(self.embeddings) > 2:\n            self.embeddings.popitem(last=False)\n        return value\n\n    def generate(self, prompt_a, prompt_b, blend, seed):\n        with self.torch.inference_mode():\n            embedding = self._embedding(prompt_a)\n            if prompt_b and blend > 0:\n                other = self._embedding(prompt_b)\n                embedding = (tuple((1 - blend) * a + blend * b for a, b in zip(embedding, other))\n                             if self.model == \'sdxl-turbo\' else (1 - blend) * embedding + blend * other)\n            conditioning = ({\'prompt_embeds\': embedding[0], \'pooled_prompt_embeds\': embedding[1]}\n                            if self.model == \'sdxl-turbo\' else {\'prompt_embeds\': embedding})\n            output = self.pipe(\n                **conditioning, num_inference_steps=1, guidance_scale=0.0,\n                width=SEED_SIZE[0], height=SEED_SIZE[1], output_type=\'np\',\n                generator=self.torch.Generator(device=self.device).manual_seed(seed)).images[0]\n        if output.shape != (SEED_SIZE[1], SEED_SIZE[0], 3) or not np.isfinite(output).all():\n            raise ValueError(f\'{self.label} returned an invalid seed image\')\n        return (np.clip(output, 0, 1) * 255).round().astype(np.uint8)\n'}
for name, content in sources.items():
    (runtime_root / name).write_text(content, encoding='utf-8')
sys.path.insert(0, str(runtime_root))
import importlib
import colab_setup
import waypoint_setup
importlib.reload(colab_setup)
importlib.reload(waypoint_setup)
colab_setup.configure_model_cache(USE_GOOGLE_DRIVE, DRIVE_MODEL_FOLDER)
# Modern Hub uses HF_HUB_CACHE; avoid the legacy Transformers cache alias.
os.environ.pop('TRANSFORMERS_CACHE', None)
waypoint_python, waypoint_tunnel = waypoint_setup.setup(runtime_root, prompt_seeds=ENABLE_PROMPT_SEEDS)
os.environ['WAYPOINT_PROMPT_SEEDS'] = '1' if ENABLE_PROMPT_SEEDS else '0'
os.environ['WAYPOINT_PROMPT_SEED_MODEL'] = PROMPT_SEED_MODEL
os.environ['TORCHINDUCTOR_CACHE_DIR'] = str(runtime_root / 'inductor-cache')
print('Waypoint environment ready. Run Step 3.')


In [ ]:
# @title Step 3: Launch Waypoint and Cloudflare
QUANTIZATION = "intw8a8" # @param ["intw8a8", "bf16"]
# @markdown Fast reduces first-frame compilation work. Max-autotune trades startup time for possible throughput.
COMPILE_PROFILE = "fast" # @param ["fast", "max-autotune"]
MAX_FPS = 20 # @param {type:"integer"}
TUNNEL_PROTOCOL = "http2" # @param ["http2", "auto", "quic"]
if 'waypoint_session' in globals():
    waypoint_session.stop()
import importlib
import colab_launcher
importlib.reload(colab_launcher)
os.environ['WAYPOINT_QUANT'] = QUANTIZATION
os.environ['WAYPOINT_COMPILE_PROFILE'] = COMPILE_PROFILE
os.environ['WAYPOINT_PROMPT_SEED_MODEL'] = PROMPT_SEED_MODEL
waypoint_session = colab_launcher.BridgeSession(
    root=runtime_root, backend='waypoint', server_script='colab_server.py',
    python_executable=waypoint_python, cloudflared=waypoint_tunnel,
    startup_timeout=1800, max_fps=MAX_FPS, tunnel_protocol=TUNNEL_PROTOCOL)
try:
    url = waypoint_session.start()
    print('Paste this URL into the updated TouchDesigner component:')
    print(url, flush=True)
    print('World page: choose Prompt Model (A/B) if enabled, or Seed World from Input.')
    waypoint_session.wait()
except KeyboardInterrupt:
    print('Stopping this Waypoint session...')
finally:
    waypoint_session.stop()
